# Market Regime Allocation with Hidden Markov Models

## Table of Contents

1. [Project Motivation](#1.-Project-Motivation)
2. [Asset Universe and Data Sources](#2.-Asset-Universe-and-Data-Sources)
3. [Feature Engineering](#3.-Feature-Engineering)
4. [Hidden Markov Model Explanation](#4.-Hidden-Markov-Model-Explanation)
5. [Regime Model Training](#5.-Regime-Model-Training)
6. [Regime Interpretation](#6.-Regime-Interpretation)
7. [Strategy Design](#7.-Strategy-Design)
8. [Walk-Forward Backtest](#8.-Walk-Forward-Backtest)

## 1. Project Motivation

Markets do not behave the same way all the time.

Sometimes the market is calm: equities trend upward, volatility is low, and investors are willing to take risk. At other times, the market becomes stressed: prices fall quickly, volatility rises, and assets that usually diversify a portfolio may stop behaving as expected.

This project is about building a systematic way to identify those changing market conditions and test whether they can support better portfolio allocation decisions.

### Simple intuition

A simple way to think about market regimes is to compare markets to weather.

If the weather is sunny, you dress differently than if there is a storm. You do not need to know the exact weather tomorrow to understand that an umbrella is useful when storm conditions are becoming more likely.

Market regimes are similar. The goal is not to predict every daily market move. The goal is to estimate the current environment, such as calm growth, high-volatility stress, or inflation/rate pressure, and then ask whether the portfolio should be positioned differently under those conditions.

### Why regimes matter

Many investment strategies implicitly assume that market relationships are reasonably stable. For example, a balanced portfolio often assumes that equities provide growth while bonds help reduce risk during equity selloffs.

In reality, these relationships can change. During some stress periods, volatility rises, risky assets fall together, and diversification becomes less reliable. During inflation or rate-hiking environments, long-duration bonds may not hedge equities as well as they do during growth shocks.

A good example is 2022. Many investors had become used to the idea that long-term Treasury bonds could help hedge equity drawdowns. This worked well in periods such as the dot-com crash and the 2008 Global Financial Crisis, where recession risk led central banks to cut interest rates. Falling interest rates usually push bond prices up, which can help offset equity losses.

In 2022, the situation was different. Inflation was high, and the Federal Reserve raised interest rates aggressively to bring inflation under control. Equities fell, but long-duration Treasury bonds also fell because yields were rising. The usual stock-bond hedge failed because the market environment was different: the dominant risk was not just recession risk, but inflation and rate pressure.

This matters because a portfolio that is appropriate in one environment may become too risky in another. A regime-aware process tries to recognize when the environment has changed instead of treating all historical periods as if they came from the same market condition.

### Research background

The idea of regime shifts is well established in economics and finance.

- Hamilton's classic Markov-switching model introduced a way to model economic time series as moving between hidden states, such as expansion and recession-like conditions. The important idea for this project is that these states are not directly observed. They are inferred from data as probabilities, not treated as objective labels. See Hamilton (1989), ["A New Approach to the Economic Analysis of Nonstationary Time Series and the Business Cycle"](https://www.econometricsociety.org/publications/econometrica/1989/03/01/new-approach-economic-analysis-nonstationary-time-series-and).
- Ang and Bekaert studied time-varying correlations and found evidence that volatility and correlations can behave differently across regimes. This supports the idea that risk itself can be state-dependent, rather than constant through time. See their NBER working paper, ["International Asset Allocation with Time-Varying Correlations"](https://www.nber.org/papers/w7056).
- Ang and Bekaert's ["How Do Regimes Affect Asset Allocation?"](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=310626) is especially relevant to this project because it connects regime shifts directly to portfolio allocation. The key motivation is not only to identify regimes, but to ask whether different regimes should lead to different portfolio choices.
- In practical portfolio management, regime thinking is useful because risk is not only about average return. It is also about drawdowns, volatility, correlation breakdowns, and whether the portfolio is exposed to the wrong risks at the wrong time.

This project applies that broad idea to an ETF allocation setting: use market and macroeconomic signals to estimate hidden market conditions, then test whether those estimated conditions can guide risk-aware allocation decisions.

### Project question

The central question is:

> Can market and macroeconomic signals be used to identify probabilistic market regimes, and can those regimes support a systematic ETF allocation strategy that improves risk-adjusted performance and drawdown control compared to simple static benchmarks?

In simpler words:

> Can we build a model that recognizes different market conditions, then use those conditions to decide when the portfolio should be more aggressive, more defensive, or more diversified?

### What this project is not trying to do

This project is not trying to prove that a machine learning model can perfectly predict the market.

It is also not trying to build a black-box trading system that automatically maximizes returns.

Instead, this project treats machine learning as a risk analysis tool. The Hidden Markov Model estimates the likely market regime. The strategy then uses transparent allocation rules for each regime. This distinction is important: the model helps describe the environment, while the allocation rule converts that environment into a portfolio decision.

### What success looks like

A useful result does not have to mean beating the stock market on raw return.

Because this is a risk-aware allocation project, success will be judged using metrics such as:

- lower maximum drawdown,
- lower volatility,
- better Sharpe or Sortino ratio,
- better behavior during stress periods,
- reasonable turnover after transaction costs,
- clear and economically interpretable regimes.

A strong project should also be honest if the model does not add value beyond simple benchmarks. In that case, the learning is still useful: it tells us that a more complex regime model must justify itself against simpler allocation rules.

## 2. Asset Universe and Data Sources

The asset universe matters because the model does not trade in isolation. Even if the regime model identifies a useful market condition, the strategy can only respond through the assets available in the portfolio.

For this project, the initial allocation universe is built around four liquid ETFs: SPY, TLT, GLD, and SHY. These ETFs are chosen because they represent different portfolio roles rather than four versions of the same risk exposure.

### Why these assets?

| ETF | Asset class | Portfolio role |
|---|---|---|
| SPY | U.S. equities | Growth and risk-on exposure |
| TLT | Long-term U.S. Treasuries | Duration exposure and potential recession hedge |
| GLD | Gold | Inflation, uncertainty, and alternative defensive exposure |
| SHY | Short-term U.S. Treasuries | Defensive cash-like exposure |

SPY represents broad U.S. equity market exposure. It is the main growth asset in the portfolio and is usually expected to do well when investors are willing to take risk.

TLT represents long-duration U.S. Treasury bonds. These bonds can perform well when growth slows and interest rates fall, but they can suffer when inflation and rising rates are the main concern.

GLD represents gold exposure. Gold does not behave like a stock or bond, so it can be useful when the market is worried about inflation, currency risk, or broader uncertainty.

SHY represents short-term U.S. Treasury exposure. It is included as a lower-volatility defensive asset, similar to a place where the strategy can reduce risk when the environment is unfavorable or unclear.

### Link to regime-based allocation

This asset universe connects directly to the regime allocation idea discussed earlier.

If regimes affect returns, volatility, and correlations, then the same portfolio may not be suitable in every environment. A calm growth regime may justify more equity exposure. A recession-like stress regime may favor Treasuries or short-duration defensive assets. An inflation or rate-pressure regime may require more caution toward long-duration bonds and may make gold or short-term Treasuries more useful.

This is why the project uses assets with different economic roles. The point is not just to identify regimes, but to test whether those regime estimates can lead to more sensible allocation choices.

### Data sources

This project uses two main data sources:

| Source | What it provides | How it is used |
|---|---|---|
| Tiingo | Historical ETF price data | Asset returns and market-based features |
| FRED | Macroeconomic and risk indicators | Macro context and regime features |

Tiingo provides the historical ETF prices needed to calculate portfolio returns. From these prices, we can engineer market features such as returns, momentum, rolling volatility, drawdowns, and correlations.

FRED provides macroeconomic and risk-related time series, such as interest rates, yield spreads, inflation measures, credit spreads, and recession indicators. These features help the model distinguish between market environments that may look similar from prices alone but are driven by different economic forces.

### Why these data sources are sufficient

The strategy is based on monthly allocation decisions, not high-frequency trading. Because of that, the project does not require intraday prices, order book data, or extremely detailed execution data.

Tiingo is sufficient for the market data side because the key raw input is adjusted historical ETF prices. Indicators such as volatility, momentum, trend, drawdown, and correlation do not need to come directly from the data provider. They can be calculated from the price history, which makes the feature definitions transparent and easier to check for lookahead bias.

FRED is sufficient for the macro side because it provides widely used public economic data. The main challenge is not finding the most complicated macro dataset, but using macro data carefully. In particular, macro features should be lagged so that the backtest does not accidentally use information that would not have been available at the time.

In short: Tiingo gives the market prices, FRED gives the economic context, and the project turns both into interpretable regime features.

## 3. Feature Engineering

Feature engineering is the step where raw data becomes useful model input.

The Hidden Markov Model does not directly understand labels such as "calm growth", "crisis", or "inflation stress". It only sees numerical columns. Those columns need to describe the market environment in a way that is both measurable and financially meaningful.

The goal of this section is to build a feature dataset that helps the model distinguish between different types of market conditions.

### Simple intuition

A useful analogy is a medical check-up.

A doctor does not judge someone's health from only one number. They may look at blood pressure, heart rate, cholesterol, symptoms, and medical history. Each measurement captures a different part of the picture.

Market regimes work similarly. A single signal, such as whether SPY is above its moving average, can be useful, but it only describes one part of the market. A richer regime model should look at several dimensions: returns, volatility, drawdowns, correlations, interest rates, credit stress, inflation, and broader macro conditions.

### Candidate feature groups

The candidate features are organized by what they are trying to measure.

| Feature group | Example features | What it tells us |
|---|---|---|
| Equity return and trend | SPY monthly return, 6-month momentum, 12-month momentum | Whether the main risky asset is rising or falling |
| Equity risk | SPY realized volatility, downside volatility, drawdown | Whether equity risk is calm or stressed |
| Cross-asset behavior | TLT return, GLD return, SPY-TLT correlation, SPY-GLD correlation | Whether diversification is working as expected |
| Volatility stress | VIX level, VIX change | Whether markets expect higher uncertainty |
| Yield curve and rates | 10Y-2Y spread, 10Y-3M spread, Fed funds rate | Growth expectations and monetary policy pressure |
| Credit stress | Baa-Treasury spread or high-yield spread | Whether investors are demanding more compensation for credit risk |
| Inflation pressure | CPI inflation, inflation trend, breakeven inflation | Whether inflation is becoming a dominant market risk |
| Macro growth | industrial production, unemployment rate | Whether the broader economy is expanding or weakening |

Not every candidate feature has to be used in the final HMM. Some features may be more useful for interpreting regimes after the model is trained. The important idea is to start with a controlled set of economically meaningful signals, then decide which ones belong in the actual model.

### Raw data needed

Most market-derived features can be calculated from ETF price history. For example, returns, momentum, volatility, drawdowns, and correlations can all be derived from adjusted prices.

The macro and risk features come from FRED. These series help identify whether the market environment is being shaped by growth risk, inflation pressure, credit stress, or monetary policy.

| Data source | Raw data to retrieve | Features created from it |
|---|---|---|
| Tiingo | daily adjusted prices for SPY, TLT, GLD, SHY | returns, momentum, realized volatility, drawdown, correlations |
| FRED | VIX, yield spreads, policy rates, inflation, credit spreads, growth indicators | volatility stress, rate pressure, inflation trend, credit stress, macro growth |

The final dataset should be monthly, because the strategy will make monthly allocation decisions.

### Implementation

The following code collects raw data from Tiingo and FRED, converts it into monthly market and macro features, and combines everything into a single feature matrix.

In [ ]:
from pathlib import Path
import os

import numpy as np
import pandas as pd
import requests
from dotenv import load_dotenv

# Display settings make notebook tables easier to read.
pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", "{:.4f}".format)

# If the notebook is opened from the notebooks folder, move one level up.
# If it is opened from the project root, keep the current folder.
current_dir = Path.cwd()
project_root = current_dir.parent if current_dir.name == "notebooks" else current_dir

# Load API keys from the local .env file.
# The keys are used by the code, but never printed in the notebook.
load_dotenv(project_root / ".env")
TIINGO_API_KEY = os.getenv("TIINGO_API_KEY")
FRED_API_KEY = os.getenv("FRED_API_KEY")

# Local cache folders prevent repeated API calls every time the notebook is rerun.
RAW_DATA_DIR = project_root / "data" / "raw"
PROCESSED_DATA_DIR = project_root / "data" / "processed"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Set this to True only when you intentionally want to refresh the API data.
REFRESH_DATA = False

ETF_PRICE_CACHE = RAW_DATA_DIR / "tiingo_etf_prices.csv"
FRED_RAW_CACHE = RAW_DATA_DIR / "fred_macro_series.csv"
FEATURE_MATRIX_CACHE = PROCESSED_DATA_DIR / "feature_matrix.csv"
ASSET_RETURNS_CACHE = PROCESSED_DATA_DIR / "asset_returns.csv"

# Start in 2005 because GLD started trading in late 2004,
# so 2005 gives us a cleaner shared ETF history.
START_DATE = "2005-01-01"
# Use an explicit analysis end date so results are reproducible.
END_DATE = "2026-05-31"

ETF_TICKERS = ["SPY", "TLT", "GLD", "SHY"]

# FRED series used as candidate macro/risk inputs.
# We may not use every series in the final HMM, but they are useful candidates.
FRED_SERIES = {
    "VIXCLS": "vix",
    "T10Y2Y": "term_spread_10y2y",
    "T10Y3M": "term_spread_10y3m",
    "FEDFUNDS": "fed_funds_rate",
    "CPIAUCSL": "cpi",
    "T10YIE": "breakeven_inflation_10y",
    "BAA10Y": "credit_spread_baa10y",
    "INDPRO": "industrial_production",
    "UNRATE": "unemployment_rate",
}

print(f"Project folder: {project_root.name}")
print(f"Tiingo key available: {TIINGO_API_KEY is not None}")
print(f"FRED key available: {FRED_API_KEY is not None}")
print(f"Refresh API data: {REFRESH_DATA}")

This setup cell defines the project paths, date range, ETF tickers, and FRED series. It also creates local cache folders. The `REFRESH_DATA` flag controls whether the notebook downloads fresh data from the APIs or reuses locally saved CSV files.

In [ ]:
def fetch_tiingo_prices(ticker, start_date=START_DATE, end_date=END_DATE):
    """Download daily adjusted close prices for one ETF from Tiingo."""
    if TIINGO_API_KEY is None:
        raise ValueError("Missing TIINGO_API_KEY. Add it to your local .env file.")

    url = f"https://api.tiingo.com/tiingo/daily/{ticker}/prices"
    headers = {"Authorization": f"Token {TIINGO_API_KEY}"}
    params = {"startDate": start_date}

    if end_date is not None:
        params["endDate"] = end_date

    response = requests.get(url, headers=headers, params=params, timeout=30)
    response.raise_for_status()

    data = response.json()
    if not data:
        raise ValueError(f"No Tiingo data returned for {ticker}.")

    df = pd.DataFrame(data)
    df["date"] = pd.to_datetime(df["date"]).dt.tz_localize(None)
    df = df.set_index("date").sort_index()

    # adjClose adjusts for dividends and splits, which is what we want for returns.
    return df["adjClose"].rename(ticker)


def fetch_fred_series(series_id, start_date=START_DATE, end_date=END_DATE):
    """Download one time series from FRED."""
    if FRED_API_KEY is None:
        raise ValueError("Missing FRED_API_KEY. Add it to your local .env file.")

    url = "https://api.stlouisfed.org/fred/series/observations"
    params = {
        "series_id": series_id,
        "api_key": FRED_API_KEY,
        "file_type": "json",
        "observation_start": start_date,
    }

    if end_date is not None:
        params["observation_end"] = end_date

    response = requests.get(url, params=params, timeout=30)
    response.raise_for_status()

    observations = response.json()["observations"]
    df = pd.DataFrame(observations)
    df["date"] = pd.to_datetime(df["date"])

    # FRED sometimes uses '.' for missing observations, so convert safely to NaN.
    df["value"] = pd.to_numeric(df["value"], errors="coerce")
    df = df.set_index("date").sort_index()

    return df["value"].rename(series_id)

These helper functions keep the API logic in one place. The Tiingo function returns one adjusted-price series for an ETF. The FRED function returns one cleaned macro series, converting missing values into `NaN` and making the date column the index.

In [ ]:
# Load cached ETF prices if available; otherwise download from Tiingo.
if ETF_PRICE_CACHE.exists() and not REFRESH_DATA:
    etf_prices = pd.read_csv(ETF_PRICE_CACHE, index_col="date", parse_dates=True)
    print("Loaded ETF prices from local cache.")
else:
    etf_prices = pd.concat(
        [fetch_tiingo_prices(ticker) for ticker in ETF_TICKERS],
        axis=1,
    ).sort_index()
    etf_prices.to_csv(ETF_PRICE_CACHE, index_label="date")
    print("Downloaded ETF prices from Tiingo and saved them locally.")

# Load cached macro data if available; otherwise download from FRED.
if FRED_RAW_CACHE.exists() and not REFRESH_DATA:
    fred_raw = pd.read_csv(FRED_RAW_CACHE, index_col="date", parse_dates=True)
    print("Loaded FRED series from local cache.")
else:
    fred_raw = pd.concat(
        [fetch_fred_series(series_id) for series_id in FRED_SERIES],
        axis=1,
    ).rename(columns=FRED_SERIES).sort_index()
    fred_raw.to_csv(FRED_RAW_CACHE, index_label="date")
    print("Downloaded FRED series and saved them locally.")

print(f"ETF price shape: {etf_prices.shape}")
print(f"FRED raw shape: {fred_raw.shape}")

display(etf_prices.tail())
display(fred_raw.tail())

This cell uses a cache-first workflow. If the local CSV files already exist, the notebook loads them directly and avoids calling the APIs again. If the files do not exist, or if `REFRESH_DATA` is set to `True`, the notebook downloads fresh data and saves a local copy.

In [ ]:
# Convert daily adjusted prices to monthly prices using the last available price each month.
monthly_prices = etf_prices.resample("ME").last()

# Monthly returns are used both as candidate features and later for backtesting.
monthly_returns = monthly_prices.pct_change()
daily_returns = etf_prices.pct_change()

market_features = pd.DataFrame(index=monthly_prices.index)

# Equity return and trend features.
market_features["spy_return_1m"] = monthly_returns["SPY"]
market_features["spy_momentum_6m"] = monthly_prices["SPY"].pct_change(6)
market_features["spy_momentum_12m"] = monthly_prices["SPY"].pct_change(12)

# Drawdown measures how far SPY is below its previous high.
market_features["spy_drawdown"] = monthly_prices["SPY"] / monthly_prices["SPY"].cummax() - 1

# Realized volatility is calculated from daily returns, then sampled at month-end.
# 63 trading days is roughly 3 months; annualization uses sqrt(252).
market_features["spy_realized_vol_3m"] = (
    daily_returns["SPY"].rolling(63).std() * np.sqrt(252)
).resample("ME").last()

# Cross-asset returns help describe whether bonds/gold are acting differently from equities.
market_features["tlt_return_1m"] = monthly_returns["TLT"]
market_features["gld_return_1m"] = monthly_returns["GLD"]
market_features["shy_return_1m"] = monthly_returns["SHY"]

# Rolling correlations help measure whether diversification is working as expected.
# 126 trading days is roughly 6 months.
market_features["spy_tlt_corr_6m"] = (
    daily_returns["SPY"].rolling(126).corr(daily_returns["TLT"])
).resample("ME").last()

market_features["spy_gld_corr_6m"] = (
    daily_returns["SPY"].rolling(126).corr(daily_returns["GLD"])
).resample("ME").last()

display(market_features.tail())

The market features are calculated from ETF prices. Monthly return measures the most recent asset performance. Momentum measures performance over longer windows. Drawdown measures how far SPY is below its previous high. Realized volatility measures how much SPY has been moving day to day. Rolling correlations measure whether Treasuries or gold are moving with or against equities.

In [ ]:
# Convert FRED series to monthly frequency using the last available observation each month.
# Forward-fill handles series that are not reported every trading day.
fred_monthly = fred_raw.resample("ME").last().ffill()

# Lag macro features by one month to reduce lookahead bias.
# This is a conservative simplification because macro data is often released after month-end.
macro_input = fred_monthly.shift(1)

macro_features = pd.DataFrame(index=macro_input.index)

# Volatility stress.
macro_features["vix_level"] = macro_input["vix"]
macro_features["vix_change_1m"] = macro_input["vix"].diff(1)

# Yield curve and policy rate conditions.
macro_features["term_spread_10y2y"] = macro_input["term_spread_10y2y"]
macro_features["term_spread_10y3m"] = macro_input["term_spread_10y3m"]
macro_features["fed_funds_rate"] = macro_input["fed_funds_rate"]
macro_features["fed_funds_change_3m"] = macro_input["fed_funds_rate"].diff(3)

# Credit stress.
macro_features["credit_spread_baa10y"] = macro_input["credit_spread_baa10y"]
macro_features["credit_spread_change_3m"] = macro_input["credit_spread_baa10y"].diff(3)

# Inflation pressure.
macro_features["inflation_yoy"] = macro_input["cpi"].pct_change(12) * 100
macro_features["inflation_trend_3m"] = macro_features["inflation_yoy"].diff(3)
macro_features["breakeven_inflation_10y"] = macro_input["breakeven_inflation_10y"]

# Macro growth context.
macro_features["industrial_production_yoy"] = macro_input["industrial_production"].pct_change(12) * 100
macro_features["unemployment_rate"] = macro_input["unemployment_rate"]
macro_features["unemployment_change_3m"] = macro_input["unemployment_rate"].diff(3)

display(macro_features.tail())

The macro features are first converted to month-end observations. They are then shifted by one month so the model does not accidentally use macro information that may not have been available at the time. The resulting features describe volatility stress, yield curve conditions, monetary policy pressure, credit stress, inflation pressure, and macro growth.

In [ ]:
# Combine market and macro features into one monthly dataset.
candidate_features = pd.concat([market_features, macro_features], axis=1).sort_index()

# Drop rows with missing values after rolling windows, percentage changes, and macro lags.
# The HMM needs a complete feature matrix with no NaNs.
feature_matrix = candidate_features.dropna().copy()

# Keep monthly returns separately for later backtesting.
# These are not all necessarily model features; they will also be used to calculate portfolio performance.
asset_returns = monthly_returns.loc[feature_matrix.index].copy()

# Save processed datasets locally so later notebook sections can reuse them.
feature_matrix.to_csv(FEATURE_MATRIX_CACHE, index_label="date")
asset_returns.to_csv(ASSET_RETURNS_CACHE, index_label="date")

print(f"Candidate feature matrix shape: {feature_matrix.shape}")
print(f"Feature sample: {feature_matrix.index.min().date()} to {feature_matrix.index.max().date()}")

display(feature_matrix.tail())

The final output of this section is `feature_matrix`, where each row is one month and each column is a candidate regime feature. Rows with missing values are removed because the HMM will require complete inputs. `asset_returns` is kept separately because it will be used later to calculate strategy and benchmark performance.

## 4. Hidden Markov Model Explanation

The previous section created the `feature_matrix`: a monthly table of market and macro signals. The next question is how to turn those signals into an estimate of the current market regime.

A Hidden Markov Model, or HMM, is useful here because it is designed for situations where we observe signals over time, but the underlying state that generated those signals is not directly visible.

### Observed signals vs hidden states

In this project, the observed signals are the features we calculated earlier:

- returns,
- momentum,
- volatility,
- drawdowns,
- correlations,
- VIX,
- yield spreads,
- credit spreads,
- inflation,
- policy rates.

The hidden states are the market regimes we are trying to infer.

There is no official label that says a particular month is definitely a "calm growth regime" or an "inflation stress regime". Those regime labels are not directly observed. They are estimated from the data.

### From Markov chains to Hidden Markov Models

This idea is closely related to Markov chains, which are often introduced in linear algebra and probability courses.

A Markov chain models a system that moves between states over time. For example, a market could move between states such as:

```text
calm -> stress -> recovery
```

The key idea is the Markov property:

> The next state depends on the current state, not the full history before it.

For example, if the market is currently calm, a Markov chain might estimate:

```text
85% chance the next month remains calm
10% chance the next month becomes stressed
5% chance the next month enters another state
```

A Hidden Markov Model builds on this idea. It still models transitions between states, but the states are hidden. Instead of directly observing the state, we observe data generated by that state.

### Simple intuition

A useful analogy is a weather station.

Suppose we cannot directly observe the true weather condition, but we can observe sensor readings:

- temperature,
- humidity,
- wind speed,
- air pressure,
- rainfall.

From these readings, we might infer whether the hidden weather condition is calm, stormy, humid, or cold.

Markets are similar. We cannot directly observe the true market regime, but we can observe market and macro signals. The HMM uses those signals to estimate which hidden market state most likely generated them.

### The two parts of an HMM

An HMM has two main pieces:

| Component | Meaning in simple terms | Market interpretation |
|---|---|---|
| Transition model | How hidden states move from one period to the next | How likely regimes are to persist or switch |
| Emission model | What observations each hidden state tends to produce | What returns, volatility, spreads, and macro signals look like in each regime |

The transition model captures the idea that regimes tend to persist. A calm market usually does not randomly become a crisis regime for one month and then immediately switch back without any pattern.

The emission model captures the idea that each regime tends to produce different observable data. For example, a stress regime may produce higher volatility, weaker equity returns, and wider credit spreads.

### HMM vs k-means clustering

A useful comparison is k-means clustering.

K-means groups observations based on similarity. If two months have similar feature values, k-means may place them in the same cluster. However, k-means treats each month independently. It does not know whether one month came before or after another.

An HMM also groups observations into hidden states, but it adds time dependence. It considers both:

- what the current data looks like, and
- what regime the market was likely in previously.

A simple way to say this is:

> K-means treats the market like a set of photographs. An HMM treats the market like a movie.

That time dimension matters because market regimes usually have persistence. If the market has been calm for several months, one slightly volatile month may raise the probability of stress, but it does not necessarily mean the entire regime has changed immediately.

### What the HMM learns

In this project, the HMM will learn three main things:

1. **State characteristics**: what each hidden regime tends to look like in terms of returns, volatility, correlations, spreads, inflation, and other features.
2. **Transition probabilities**: how likely the market is to stay in the same regime or move to another regime.
3. **Regime probabilities**: for each month, the probability that the market belongs to each hidden state.

This last point is important. The HMM does not produce objective market labels. It produces probabilistic estimates based on the data and model assumptions.

For example, the model may estimate that a particular month has:

```text
10% probability of Regime 0
20% probability of Regime 1
70% probability of Regime 2
```

We may later interpret Regime 2 as an inflation or rate-pressure regime, but that interpretation comes after studying the data associated with the hidden state.

### How this connects to the project

The HMM sits between feature engineering and portfolio allocation.

The flow is:

```text
feature_matrix
    -> HMM
        -> regime probabilities
            -> regime interpretation
                -> allocation rule
                    -> backtest
```

The model does not directly decide the portfolio weights. Instead, it estimates the likely market regime. After that, we interpret the regimes and use transparent allocation rules to decide how much to hold in SPY, TLT, GLD, and SHY.

This keeps the project aligned with the risk analytics goal: the HMM is used to understand the market environment, not as a black-box return prediction machine.

## 5. Regime Model Training

The previous section explained what an HMM is conceptually. This section turns that idea into a trained model.

The goal is not to assume the market has a fixed number of regimes upfront. Instead, we train three candidate HMMs with 2, 3, and 4 hidden states, then compare whether the extra complexity seems justified.

At this stage, the regimes are still just numbered hidden states. We will interpret what those states mean economically in the next section.

### Load the modelling data

We start from the two processed files created in the feature engineering section. The `feature_matrix` contains the market and macro signals used by the HMM. The `asset_returns` table is loaded as well, but it is not used to train the HMM. It is kept for later, when we test whether regime-aware allocation rules improve portfolio behaviour.

In [ ]:
from pathlib import Path
import logging

# hmmlearn can log tiny numerical convergence messages for random starts that are not kept.
logging.getLogger("hmmlearn").setLevel(logging.ERROR)

import numpy as np
import pandas as pd

from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

# Resolve the project root whether the notebook is opened from the repo root or the notebooks folder.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"
FEATURE_MATRIX_CACHE = PROCESSED_DATA_DIR / "feature_matrix.csv"
ASSET_RETURNS_CACHE = PROCESSED_DATA_DIR / "asset_returns.csv"
MODEL_COMPARISON_CACHE = PROCESSED_DATA_DIR / "regime_model_comparison.csv"

if not FEATURE_MATRIX_CACHE.exists() or not ASSET_RETURNS_CACHE.exists():
    raise FileNotFoundError(
        "Processed data was not found. Run the feature engineering section first "
        "to create feature_matrix.csv and asset_returns.csv."
    )

feature_matrix = pd.read_csv(FEATURE_MATRIX_CACHE, parse_dates=["date"], index_col="date")
asset_returns = pd.read_csv(ASSET_RETURNS_CACHE, parse_dates=["date"], index_col="date")

# Sorting protects us from accidental date-order issues after loading from CSV.
feature_matrix = feature_matrix.sort_index()
asset_returns = asset_returns.sort_index()

print(f"Feature matrix shape: {feature_matrix.shape}")
print(f"Asset returns shape: {asset_returns.shape}")
print(f"Sample period: {feature_matrix.index.min().date()} to {feature_matrix.index.max().date()}")

display(feature_matrix.head())

This code reloads the clean monthly datasets from the local cache. The HMM only receives the feature matrix, because it is trying to infer market conditions from signals such as returns, volatility, yield spreads, inflation, credit stress, and unemployment. Asset returns are deliberately kept separate so that later performance testing does not accidentally become part of model training.

### Split the sample by time

For normal machine learning examples, it is common to randomly split rows into training and test sets. That would be inappropriate here because financial data has a time order. A random split would allow the model setup to learn from future periods while pretending to evaluate the past.

Instead, we split the data chronologically:

```text
Training period: used to fit the HMM parameters
Validation period: used to compare the 2-state, 3-state, and 4-state candidates
Test period: left aside for the later backtest section
```

The training period runs from 2006 to 2018, so it includes the global financial crisis, the post-crisis recovery, the eurozone debt stress period, the low-rate quantitative easing environment, and the volatility/rate-hike stress of 2018. The validation period covers 2019 to 2021, including the COVID crash, the rapid policy response, the reopening period, and the early phase of inflation pressure. The test period begins in 2022, which includes the stock-bond drawdown, aggressive rate hikes, persistent inflation stress, and later geopolitical and macro uncertainty.

This is still not the final trading simulation. The fully realistic version will be the walk-forward backtest later, where the model is repeatedly refit using only information available at that point in time.

In [ ]:
TRAIN_END = pd.Timestamp("2018-12-31")
VALIDATION_END = pd.Timestamp("2021-12-31")

train_features = feature_matrix.loc[feature_matrix.index <= TRAIN_END].copy()
validation_features = feature_matrix.loc[
    (feature_matrix.index > TRAIN_END) & (feature_matrix.index <= VALIDATION_END)
].copy()
test_features = feature_matrix.loc[feature_matrix.index > VALIDATION_END].copy()

split_summary = pd.DataFrame(
    {
        "start": [train_features.index.min(), validation_features.index.min(), test_features.index.min()],
        "end": [train_features.index.max(), validation_features.index.max(), test_features.index.max()],
        "months": [len(train_features), len(validation_features), len(test_features)],
    },
    index=["train", "validation", "test"],
)

print(split_summary)

The split keeps the timeline intact. The training period is where the model learns the statistical behaviour of the hidden states. The validation period helps us compare candidate regime counts without touching the later test period. The test period is left untouched because it will be more meaningful in the walk-forward backtest.

### Standardise the features without using future information

The feature columns are measured in different units. Monthly returns, VIX levels, inflation changes, yield spreads, and unemployment rates do not naturally live on the same scale. If we feed them into the HMM directly, large-number variables can dominate smaller-number variables.

To avoid this, we standardise each feature by subtracting its mean and dividing by its standard deviation. The important detail is that the scaler is fitted on the training period only, then reused on validation and test data.

In [ ]:
scaler = StandardScaler()

# Fit the scaler only on the training period to avoid using future information.
X_train = scaler.fit_transform(train_features)
X_validation = scaler.transform(validation_features)
X_test = scaler.transform(test_features)
X_all = scaler.transform(feature_matrix)

scaled_train_features = pd.DataFrame(X_train, index=train_features.index, columns=train_features.columns)
scaled_validation_features = pd.DataFrame(X_validation, index=validation_features.index, columns=validation_features.columns)
scaled_test_features = pd.DataFrame(X_test, index=test_features.index, columns=test_features.columns)
scaled_all_features = pd.DataFrame(X_all, index=feature_matrix.index, columns=feature_matrix.columns)

# After standardisation, training features should have mean near 0 and standard deviation near 1.
scaling_check = pd.DataFrame(
    {
        "train_mean": scaled_train_features.mean().round(3),
        "train_std": scaled_train_features.std(ddof=0).round(3),
    }
)

display(scaling_check.head(10))

This cell puts all model inputs onto a comparable scale. The scaler is fitted only on the training period and then reused on the validation and test periods, which avoids letting future market conditions influence how past data is transformed.

### Define model comparison helpers

We will train Gaussian HMMs. This means each hidden state is assumed to generate feature values from a normal distribution. We also use diagonal covariance matrices, which is a practical choice for a modest monthly dataset because it keeps the number of parameters manageable.

Each candidate model is trained several times with different random seeds. This matters because HMM training can start from different initial guesses and occasionally settle into weaker solutions. To keep the setup transparent, we initialise each model with simple starting values instead of relying on hidden library defaults.

In [ ]:
STATE_COUNTS = [2, 3, 4]
RANDOM_SEEDS = range(20)
N_ITER = 1000
TOLERANCE = 1e-3


def count_hmm_parameters(n_states, n_features):
    """Count free parameters for a Gaussian HMM with diagonal covariance."""
    start_prob_params = n_states - 1
    transition_params = n_states * (n_states - 1)
    mean_params = n_states * n_features
    variance_params = n_states * n_features
    return start_prob_params + transition_params + mean_params + variance_params


def calculate_information_criteria(log_likelihood, n_states, n_features, n_observations):
    """Calculate AIC and BIC so models with different state counts can be compared."""
    n_params = count_hmm_parameters(n_states, n_features)
    aic = -2 * log_likelihood + 2 * n_params
    bic = -2 * log_likelihood + n_params * np.log(n_observations)
    return aic, bic, n_params


def fit_hmm_candidate(n_states, random_seed, X):
    """Fit one HMM candidate for a given state count and random seed."""
    rng = np.random.default_rng(random_seed)

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=N_ITER,
        tol=TOLERANCE,
        min_covar=1e-3,
        random_state=random_seed,
        init_params="",  # We set initial values ourselves so the setup is explicit.
    )

    # Start with equal regime probabilities.
    model.startprob_ = np.repeat(1 / n_states, n_states)

    # Start with persistent regimes: high probability of staying in the same state.
    off_diagonal_probability = 0.05 / (n_states - 1)
    model.transmat_ = np.full((n_states, n_states), off_diagonal_probability)
    np.fill_diagonal(model.transmat_, 0.95)

    # Use random training months as initial state means.
    initial_mean_rows = rng.choice(len(X), size=n_states, replace=False)
    model.means_ = X[initial_mean_rows].copy()

    # Start every state with the overall feature variance.
    model.covars_ = np.tile(np.var(X, axis=0) + 1e-3, (n_states, 1))

    model.fit(X)
    return model

The helper functions keep the model training code readable. `count_hmm_parameters` and `calculate_information_criteria` are used because a model with more states will usually fit the training data better, but it also has more parameters. AIC and BIC add a penalty for extra complexity, which helps us avoid choosing a more complicated model just because it memorises the sample better.

### Train 2-state, 3-state, and 4-state HMMs

Now we fit the candidate models. For each number of hidden states, we try multiple random seeds and keep the strongest run. We record both statistical fit and practical diagnostics, such as whether one state is used for only a tiny fraction of the sample.

A regime model is not useful just because it has a high likelihood. It also needs to produce states that are stable enough and common enough to interpret.

In [ ]:
model_results = []
best_models = {}

n_features = X_train.shape[1]
n_train_observations = X_train.shape[0]

for n_states in STATE_COUNTS:
    best_model = None
    best_train_log_likelihood = -np.inf

    for seed in RANDOM_SEEDS:
        model = fit_hmm_candidate(n_states=n_states, random_seed=seed, X=X_train)
        train_log_likelihood = model.score(X_train)

        # Keep the best random initialisation for this state count.
        if train_log_likelihood > best_train_log_likelihood:
            best_train_log_likelihood = train_log_likelihood
            best_model = model

    train_log_likelihood = best_model.score(X_train)
    validation_log_likelihood = best_model.score(X_validation)
    aic, bic, n_params = calculate_information_criteria(
        log_likelihood=train_log_likelihood,
        n_states=n_states,
        n_features=n_features,
        n_observations=n_train_observations,
    )

    train_states = best_model.predict(X_train)
    state_counts = pd.Series(train_states).value_counts(normalize=True).sort_index()
    min_state_share = state_counts.min()
    max_state_share = state_counts.max()

    # The diagonal of the transition matrix shows how likely each state is to persist.
    average_self_transition = np.diag(best_model.transmat_).mean()

    model_results.append(
        {
            "n_states": n_states,
            "best_train_log_likelihood": train_log_likelihood,
            "train_log_likelihood_per_month": train_log_likelihood / len(X_train),
            "validation_log_likelihood": validation_log_likelihood,
            "validation_log_likelihood_per_month": validation_log_likelihood / len(X_validation),
            "aic": aic,
            "bic": bic,
            "n_parameters": n_params,
            "converged": best_model.monitor_.converged,
            "iterations": best_model.monitor_.iter,
            "min_state_share_train": min_state_share,
            "max_state_share_train": max_state_share,
            "average_self_transition": average_self_transition,
        }
    )
    best_models[n_states] = best_model

model_comparison = pd.DataFrame(model_results).sort_values("n_states")
model_comparison.to_csv(MODEL_COMPARISON_CACHE, index=False)

display(model_comparison.round(4))

This table is the first serious comparison between candidate regime counts.

A few rules of thumb help interpret the columns:

- Higher training log likelihood means the model fits the training period better.
- Higher validation log likelihood is more important because it suggests the model explains later unseen data better.
- Lower AIC and BIC are better because they reward fit but penalise extra parameters.
- `min_state_share_train` should not be too close to zero; otherwise, the model may have created a regime that is barely used.
- `average_self_transition` measures persistence. Higher values mean regimes tend to last longer instead of switching every month.

In this run, the 4-state model fits the training period best, which is expected because it has more flexibility. However, the 3-state model has the strongest validation log likelihood, while the 4-state validation result is weaker. That is an early warning that the 4-state version may be overfitting. We still do not choose the final model from this table alone; the next section will check whether the states make economic sense.

### Generate diagnostic regime probabilities

The most useful output of an HMM is not only the single most likely state. It is the probability assigned to each hidden state each month.

For example, a month might be 80% likely to belong to a stressed regime and 20% likely to belong to a normal regime. That is more informative than pretending the market condition is known with certainty.

The probabilities below are diagnostic model outputs for interpretation. The final trading backtest will still need a walk-forward design so that allocation decisions only use information available at the time.

In [ ]:
regime_probability_outputs = {}
probability_diagnostics = []

for n_states, model in best_models.items():
    probabilities = model.predict_proba(X_all)
    probability_columns = [f"state_{state}_probability" for state in range(n_states)]

    regime_probabilities = pd.DataFrame(
        probabilities,
        index=feature_matrix.index,
        columns=probability_columns,
    )

    # The most likely state is a convenient summary, but the full probability distribution is more informative.
    regime_probabilities["most_likely_state"] = np.argmax(probabilities, axis=1)
    regime_probability_outputs[n_states] = regime_probabilities

    output_path = PROCESSED_DATA_DIR / f"regime_probabilities_{n_states}_states.csv"
    regime_probabilities.to_csv(output_path, index_label="date")

    max_probability = regime_probabilities[probability_columns].max(axis=1)
    state_shares = regime_probabilities["most_likely_state"].value_counts(normalize=True).sort_index()

    for state, share in state_shares.items():
        probability_diagnostics.append(
            {
                "n_states": n_states,
                "diagnostic": f"state_{state}_share",
                "value": share,
            }
        )

    probability_diagnostics.extend(
        [
            {
                "n_states": n_states,
                "diagnostic": "average_highest_probability",
                "value": max_probability.mean(),
            },
            {
                "n_states": n_states,
                "diagnostic": "lowest_highest_probability",
                "value": max_probability.min(),
            },
        ]
    )

probability_diagnostics = pd.DataFrame(probability_diagnostics)

# Use the 3-state candidate as an example because it is often the most interpretable middle ground.
example_probabilities = regime_probability_outputs[3].copy()
example_probability_columns = [
    column for column in example_probabilities.columns if column.endswith("_probability")
]
example_probabilities["highest_probability"] = example_probabilities[example_probability_columns].max(axis=1)
least_certain_examples = example_probabilities.sort_values("highest_probability").head(10)

print("Saved diagnostic regime probability files:")
for n_states in STATE_COUNTS:
    print(f"- regime_probabilities_{n_states}_states.csv")

display(probability_diagnostics.pivot(index="diagnostic", columns="n_states", values="value").round(4))
display(least_certain_examples.round(4))

This cell saves the monthly regime probabilities for the 2-state, 3-state, and 4-state candidates. The state numbers are just model labels for now. `state_0` does not automatically mean good or bad; we need to examine the feature behaviour and asset returns associated with each state before attaching economic meaning.

The second table shows months where the 3-state model is least certain. These examples are useful because they make it clear that the HMM produces probabilities, not just hard regime labels.

That interpretation step comes next.

## 6. Regime Interpretation

The previous section trained several candidate HMMs. This section asks what the inferred states actually mean.

An HMM only produces numbered states. It does not know whether `state_0` is a calm market, a crisis market, or an inflation-stress market. Those labels have to come from evidence. We therefore study each state using asset returns, volatility, drawdowns, macro conditions, frequency, persistence, and historical timing before assigning human-readable labels.

### Select the model to interpret

Part 5 compared 2-state, 3-state, and 4-state HMMs. For interpretation, this project uses the 3-state model as the primary candidate.

In the current model comparison, the 3-state HMM has the strongest validation log likelihood per month at `-54.1801`, compared with `-55.1390` for the 2-state model and `-62.8021` for the 4-state model. The 4-state model has the best training log likelihood per month at `-20.9119`, which is expected because a more flexible model can fit the training sample more closely. However, its weaker validation performance suggests that the extra complexity may not generalise as well.

Log likelihood measures how plausible the observed data is under the fitted model. Higher values are better because they mean the model assigns more probability to the data it is asked to explain. Negative values are normal here because probabilities are between 0 and 1, and the logarithm of a number between 0 and 1 is negative. The exact value is not very meaningful on its own; it is mainly useful when comparing models fitted on the same data and evaluated over the same validation period.

The reason for selecting the 3-state model is therefore practical rather than absolute. The 2-state model is simple but may be too coarse because it mainly separates normal and stressed environments. The 4-state model fits the training period better, but its weaker validation performance suggests possible overfitting. The 3-state model offers a middle ground: it is richer than a simple calm/stress split while remaining easier to interpret than the 4-state version.

This choice can still be revisited later if the regime evidence does not make economic sense.

In [ ]:
SELECTED_N_STATES = 3

REGIME_PROBABILITY_CACHE = PROCESSED_DATA_DIR / f"regime_probabilities_{SELECTED_N_STATES}_states.csv"

if not MODEL_COMPARISON_CACHE.exists() or not REGIME_PROBABILITY_CACHE.exists():
    raise FileNotFoundError(
        "Regime model outputs were not found. Run the regime model training section first."
    )

model_comparison = pd.read_csv(MODEL_COMPARISON_CACHE)
regime_probabilities = pd.read_csv(
    REGIME_PROBABILITY_CACHE,
    parse_dates=["date"],
    index_col="date",
).sort_index()

selected_model_row = model_comparison.loc[
    model_comparison["n_states"] == SELECTED_N_STATES
].copy()

print(f"Selected model: {SELECTED_N_STATES}-state HMM")
display(selected_model_row.round(4))
display(regime_probabilities.head())

This cell fixes the interpretation section to one selected model: the 3-state HMM. This is important for reproducibility. We still keep the 2-state and 4-state results from Part 5, but the interpretation tables below all refer to the same selected regime model.

### Combine regimes with features and asset returns

To interpret the regimes, we need one table that contains the HMM output, the market and macro features, and the asset returns. This lets us ask questions such as: when the model assigns a month to `state_1`, what were equity returns, bond returns, volatility, inflation, and credit spreads usually doing?

In [ ]:
# Prefix asset return columns so they are easy to distinguish from engineered features.
asset_returns_for_analysis = asset_returns.add_prefix("asset_")

regime_analysis = (
    feature_matrix
    .join(asset_returns_for_analysis, how="inner")
    .join(regime_probabilities, how="inner")
)

# Rename the hard regime assignment to make later groupby code easier to read.
regime_analysis = regime_analysis.rename(columns={"most_likely_state": "regime_state"})
regime_analysis["regime_state"] = regime_analysis["regime_state"].astype(int)

print(f"Regime analysis dataset shape: {regime_analysis.shape}")
print(f"Sample period: {regime_analysis.index.min().date()} to {regime_analysis.index.max().date()}")
print(f"Missing values: {regime_analysis.isna().sum().sum()}")

display(regime_analysis.head())

This combined dataset is the base table for interpretation. Each row is one month. The feature columns describe the market and macro environment, the asset return columns show what each ETF did that month, and `regime_state` is the most likely hidden state assigned by the HMM.

### Study regime frequency and persistence

Before looking at returns or macro conditions, we should check whether the regimes are usable. A useful regime should appear often enough to study and should not switch randomly every month.

Frequency tells us how common each state is. Persistence tells us whether states tend to last for several months at a time.

In [ ]:
def calculate_regime_runs(regime_series):
    """Return consecutive runs of the same regime state."""
    regime_series = regime_series.sort_index()
    run_id = regime_series.ne(regime_series.shift()).cumsum()

    runs = (
        pd.DataFrame({"regime_state": regime_series, "run_id": run_id})
        .groupby("run_id")
        .agg(
            regime_state=("regime_state", "first"),
            start_date=("regime_state", lambda values: values.index.min()),
            end_date=("regime_state", lambda values: values.index.max()),
            duration_months=("regime_state", "size"),
        )
        .reset_index(drop=True)
    )
    return runs

regime_counts = regime_analysis["regime_state"].value_counts().sort_index()
regime_frequency = pd.DataFrame(
    {
        "months": regime_counts,
        "share_of_sample": regime_counts / len(regime_analysis),
    }
)

regime_runs = calculate_regime_runs(regime_analysis["regime_state"])
regime_persistence = (
    regime_runs
    .groupby("regime_state")
    .agg(
        number_of_runs=("duration_months", "count"),
        average_duration_months=("duration_months", "mean"),
        median_duration_months=("duration_months", "median"),
        max_duration_months=("duration_months", "max"),
    )
)

print("Regime frequency")
display(regime_frequency.round(4))

print("Regime persistence")
display(regime_persistence.round(2))

The frequency table shows how much of the sample belongs to each state. The persistence table groups consecutive months together, so a six-month stretch in the same state is counted as one six-month run. If a state appears frequently and has reasonable duration, it is easier to interpret and more practical to use in a strategy.

The current output suggests that Regime 0 and Regime 1 are the two dominant states. Regime 0 covers `106` months, or about `44.0%` of the sample, across `4` separate runs with an average duration of `26.5` months. Regime 1 covers `87` months, or about `36.1%` of the sample, across `3` runs with an average duration of `29.0` months. Together, these two regimes account for roughly `80.1%` of the full sample and appear to behave like persistent background environments.

Regime 2 is different. It covers only `48` months, or about `19.9%` of the sample, but appears across `6` separate runs with an average duration of only `8.0` months. A run means one uninterrupted episode in the same regime, so a higher number of shorter runs suggests that Regime 2 is more episodic. At this stage, that pattern hints that Regime 2 may represent temporary stress or dislocation periods, but we should confirm that using the return, market-risk, macro, and historical timing evidence below.

### Study asset returns by regime

Because the final project will allocate across SPY, TLT, GLD, and SHY, asset behaviour by regime is central. We want to know which assets tend to perform well, which assets are volatile, and which assets suffer the worst months in each inferred state.

In [ ]:
asset_return_columns = [
    "asset_SPY",
    "asset_TLT",
    "asset_GLD",
    "asset_SHY",
]

asset_return_summary = []

for regime_state, group in regime_analysis.groupby("regime_state"):
    for column in asset_return_columns:
        monthly_returns = group[column]
        monthly_mean = monthly_returns.mean()
        monthly_volatility = monthly_returns.std()

        asset_return_summary.append(
            {
                "regime_state": regime_state,
                "asset": column.replace("asset_", ""),
                "average_monthly_return": monthly_mean,
                "annualized_return_approx": monthly_mean * 12,
                "annualized_volatility": monthly_volatility * np.sqrt(12),
                "best_month": monthly_returns.max(),
                "worst_month": monthly_returns.min(),
                "positive_month_share": (monthly_returns > 0).mean(),
            }
        )

asset_return_summary = pd.DataFrame(asset_return_summary)

display(asset_return_summary.round(4))

This table shows how each ETF behaved inside each inferred regime. The most important columns to focus on are `average_monthly_return`, `annualized_volatility`, `worst_month`, and `positive_month_share`.

Regime 0 looks like the strongest risk-asset environment. SPY has an average monthly return of `1.40%`, or about `16.85%` annualized, and is positive in `76.4%` of months. GLD also performs strongly in this state, with an average monthly return of `1.33%`, or about `15.94%` annualized. This suggests that Regime 0 is not simply a defensive environment; it is a favourable return environment for risk assets, with relatively controlled equity volatility.

Regime 1 is also positive for equities, but less broad. SPY averages `1.20%` per month, or about `14.44%` annualized, and is positive in `65.5%` of months. TLT and GLD are positive but weaker, while SHY is close to flat. This looks like an environment where equities still work, but the cross-asset return profile is less strong than Regime 0.

Regime 2 is the clear stress candidate from the asset-return perspective. SPY has a negative average monthly return of `-0.36%`, or about `-4.27%` annualized, and its annualized volatility jumps to `24.65%`. Its worst month is `-16.52%`, much worse than the worst SPY months in Regime 0 and Regime 1. GLD is the strongest asset in this state, with an average monthly return of `1.13%`, while TLT and SHY remain positive on average. This suggests that Regime 2 may be the state where equity risk is most problematic and defensive or diversifying assets become more important.

### Study market-risk features by regime

Next we inspect market-risk features such as SPY momentum, drawdown, volatility, and cross-asset correlations. These features help distinguish a calm growth regime from a falling-market regime or a diversification-breakdown regime.

In [ ]:
market_feature_columns = [
    "spy_return_1m",
    "spy_momentum_6m",
    "spy_momentum_12m",
    "spy_drawdown",
    "spy_realized_vol_3m",
    "spy_tlt_corr_6m",
    "spy_gld_corr_6m",
]

market_feature_summary = (
    regime_analysis
    .groupby("regime_state")[market_feature_columns]
    .agg(["mean", "median"])
)

# Compare each regime mean against the full-sample mean in standard deviation units.
# This makes features with different units easier to compare directionally.
market_feature_zscores = (
    regime_analysis.groupby("regime_state")[market_feature_columns].mean()
    - regime_analysis[market_feature_columns].mean()
) / regime_analysis[market_feature_columns].std()

print("Market feature summary")
display(market_feature_summary.round(4))

print("Market feature z-scores vs full sample")
display(market_feature_zscores.round(2))

The first table shows the actual mean and median values by regime. The second table is usually easier to scan because it converts each feature into a z-score relative to the full sample. A z-score of `0.52`, for example, means the regime average is `0.52` standard deviations above the full-sample average. A z-score of `-1.32` means it is `1.32` standard deviations below the full-sample average.

For these market-risk features, the z-score table is the main table to focus on because the features have different units. Returns, drawdowns, volatility, and correlations cannot be compared directly in raw units, but z-scores make the direction and relative unusualness easier to read.

Regime 0 has shallow drawdowns and calmer risk conditions. Its SPY drawdown z-score is `0.47`, which means drawdowns are less severe than usual because drawdown values are closer to zero. Realized volatility is below average at `-0.38`. SPY-TLT correlation is higher than usual at `0.52`, so the equity-bond relationship is less negatively correlated than in some other regimes.

Regime 1 has the strongest equity trend profile. SPY 12-month momentum is `0.48` standard deviations above average and 6-month momentum is `0.30` above average. SPY-TLT correlation is `-0.51`, meaning stocks and long-term Treasuries are more negatively correlated than usual in this state. That is important because negative stock-bond correlation can make diversification more effective.

Regime 2 is the clearest market-stress state. SPY 12-month momentum is `-1.32`, 6-month momentum is `-0.96`, and SPY drawdown is `-1.16`, meaning equities are much weaker and drawdowns are much deeper than usual. Realized volatility is `1.29` standard deviations above average. This confirms what the asset-return table suggested: Regime 2 is where equity risk is most elevated.

### Study macro conditions by regime

A regime should not only describe price action. Because this project includes macroeconomic features, we also inspect whether each state is associated with different inflation, policy-rate, credit, growth, and labour-market conditions.

In [ ]:
macro_feature_columns = [
    "vix_level",
    "vix_change_1m",
    "term_spread_10y2y",
    "term_spread_10y3m",
    "fed_funds_rate",
    "fed_funds_change_3m",
    "credit_spread_baa10y",
    "credit_spread_change_3m",
    "inflation_yoy",
    "inflation_trend_3m",
    "breakeven_inflation_10y",
    "industrial_production_yoy",
    "unemployment_rate",
    "unemployment_change_3m",
]

macro_feature_summary = (
    regime_analysis
    .groupby("regime_state")[macro_feature_columns]
    .agg(["mean", "median"])
)

macro_feature_zscores = (
    regime_analysis.groupby("regime_state")[macro_feature_columns].mean()
    - regime_analysis[macro_feature_columns].mean()
) / regime_analysis[macro_feature_columns].std()

print("Macro feature summary")
display(macro_feature_summary.round(4))

print("Macro feature z-scores vs full sample")
display(macro_feature_zscores.round(2))

The macro summary helps identify whether a regime is linked to high volatility, credit stress, inflation pressure, monetary tightening, weak production, or labour-market weakness. As with the market-risk section, the z-score table is usually the best place to focus because it shows what is unusually high or low relative to the full sample.

Regime 0 looks like a low-stress, higher-policy-rate environment. VIX is below average with a z-score of `-0.51`, credit spreads are below average at `-0.64`, and unemployment is much lower than usual at `-0.73`. Fed funds is above average at `0.79`, which suggests this state often appears when policy rates are higher but markets are not under obvious stress.

Regime 1 looks like an easy-policy recovery or expansion environment. Fed funds is far below average at `-0.82`, while term spreads are strongly positive with z-scores of `0.91` for the 10-year minus 2-year spread and `0.80` for the 10-year minus 3-month spread. Unemployment is above average at `0.73`, but unemployment change is below average at `-0.22`, which suggests labour conditions may be improving rather than deteriorating. This fits with periods where policy is easy and the economy is recovering.

Regime 2 has the strongest macro stress profile. VIX is `1.29` standard deviations above average, credit spreads are `0.96` above average, industrial production growth is `-1.00` below average, and unemployment change is `0.42` above average. Inflation is also above average at `0.49`. This combination points to a regime associated with stress, weaker growth, and inflation pressure rather than a normal calm environment.

### Inspect historical timing

Numbers are useful, but historical timing is the sanity check. We should ask whether the states appear during periods that make economic sense. For example, a stress-like regime should plausibly appear around major selloffs, while an inflation/rate-pressure regime might appear around 2022.

In [ ]:
historical_regime_timeline = regime_analysis[["regime_state"]].copy()
historical_regime_timeline["year"] = historical_regime_timeline.index.year

months_by_year = pd.crosstab(
    historical_regime_timeline["year"],
    historical_regime_timeline["regime_state"],
)

# For each regime, show the longest continuous periods assigned to that state.
longest_regime_runs = (
    regime_runs
    .sort_values(["regime_state", "duration_months"], ascending=[True, False])
    .groupby("regime_state")
    .head(5)
    .sort_values(["regime_state", "duration_months"], ascending=[True, False])
)

print("Months by year and regime")
display(months_by_year)

print("Longest continuous regime runs")
display(longest_regime_runs)

The year-by-regime table shows when each state appears across the historical sample. The longest-run table shows whether regimes persist through recognisable periods. This helps guard against assigning labels from summary statistics alone.

The historical timing strongly supports the interpretation of Regime 2 as an episodic stress state. It captures all `12` months of 2008 and `9` months of 2009, covering the global financial crisis period. It also captures `9` months of 2020 around the COVID shock and policy response, and `11` months of 2022 during the inflation, rate-hiking, and stock-bond drawdown period.

Regime 1 appears heavily in the post-crisis and easy-policy period. It covers all `12` months of each year from 2012 to 2015, and then all `12` months of 2021. This lines up with the macro evidence: low policy rates, positive term spreads, and improving conditions after stress periods.

Regime 0 appears in calmer expansion periods such as 2006-2007, much of 2016-2019, and 2023-2026 so far. The long run from March 2023 to May 2026 lasts `39` months in the current sample, while the 2016-2018 run lasts `33` months. This supports the view that Regime 0 is a persistent, relatively favourable market environment rather than a short-lived shock state.

### Build an interpretation table

Now we combine the evidence into a compact interpretation table. The labels below are still evidence-based descriptions rather than permanent truths. They are useful because Part 7 needs human-readable regimes before we can design transparent allocation rules.

In [ ]:
def top_zscore_features(zscore_table, regime_state, n=5):
    """Return the most unusually high and low features for one regime."""
    regime_scores = zscore_table.loc[regime_state].dropna().sort_values()
    low_features = regime_scores.head(n).index.tolist()
    high_features = regime_scores.tail(n).sort_values(ascending=False).index.tolist()
    return low_features, high_features

combined_zscores = pd.concat(
    [
        market_feature_zscores,
        macro_feature_zscores,
    ],
    axis=1,
)

interpretation_rows = []

for regime_state in sorted(regime_analysis["regime_state"].unique()):
    low_features, high_features = top_zscore_features(combined_zscores, regime_state)
    best_assets = (
        asset_return_summary.loc[asset_return_summary["regime_state"] == regime_state]
        .sort_values("average_monthly_return", ascending=False)["asset"]
        .head(2)
        .tolist()
    )
    worst_assets = (
        asset_return_summary.loc[asset_return_summary["regime_state"] == regime_state]
        .sort_values("average_monthly_return")["asset"]
        .head(2)
        .tolist()
    )

    interpretation_rows.append(
        {
            "regime_state": regime_state,
            "sample_share": regime_frequency.loc[regime_state, "share_of_sample"],
            "average_duration_months": regime_persistence.loc[regime_state, "average_duration_months"],
            "best_assets_by_average_return": ", ".join(best_assets),
            "weakest_assets_by_average_return": ", ".join(worst_assets),
            "unusually_high_features": ", ".join(high_features),
            "unusually_low_features": ", ".join(low_features),
        }
    )

regime_interpretation_table = pd.DataFrame(interpretation_rows)

display(regime_interpretation_table.round(4))

This interpretation table pulls together frequency, persistence, asset behaviour, and the most distinctive market or macro features. It gives us the evidence needed to name the regimes. The labels should be based on the full pattern, not on a single feature.

The table points toward three economically different states. Regime 0 combines strong SPY and GLD returns, low VIX, low credit spreads, low unemployment, and relatively shallow drawdowns. A reasonable preliminary interpretation is a calm risk-on or resilient growth environment.

Regime 1 combines positive equity momentum, low policy rates, steep term spreads, higher but improving unemployment, and more negative stock-bond correlation. A reasonable preliminary interpretation is an easy-policy recovery or expansion environment, where equities perform well and bonds may still provide diversification.

Regime 2 combines negative SPY returns, high equity volatility, deep drawdowns, high VIX, wider credit spreads, weak industrial production, and stress-heavy historical periods such as 2008, 2020, and 2022. A reasonable preliminary interpretation is a crisis, stress, or macro-dislocation regime. This is likely the regime that will matter most for defensive allocation rules in Part 7.

### Assign evidence-based regime labels

Based on the current model output, the next step is to read the interpretation table together with the detailed return, feature, macro, and historical timing summaries. Only then should each state receive a human-readable label.

A good label should be:

- descriptive rather than dramatic,
- grounded in several pieces of evidence,
- useful for designing allocation rules,
- easy to explain to someone reading the project.

For example, possible labels might include `Risk-on / growth`, `Crisis / defensive stress`, or `Inflation / rate stress`, but the final labels should follow the evidence shown above rather than being assumed in advance.

This interpretation is descriptive rather than a strategy-performance result. Any claim about portfolio returns, Sharpe ratio, or drawdown should be evaluated separately using the held-out test period and the later walk-forward backtest.

## 7. Strategy Design

The regime model tells us what market condition the data most likely resembles. It does not automatically tell us how much SPY, TLT, GLD, or SHY to hold. That second step is a portfolio policy decision.

This distinction is important. The HMM makes the regime identification less dependent on hand-written labels such as "bull market" or "inflation shock". The allocation rule still needs human judgment because a portfolio has objectives and constraints: how much risk to take, whether leverage is allowed, how often to rebalance, and how aggressively to respond to uncertain signals.

The goal of this section is therefore not to find the perfect allocation. The goal is to define a transparent, defensible, and testable policy that can be evaluated later in the backtest.

### Evidence used for the allocation rule

The allocation rule is based mainly on the regime evidence from Part 6, rather than on a separate optimisation exercise.

The model found three economically interpretable states:

- **Regime 0:** SPY had an average monthly return of about **1.40%**, GLD had an average monthly return of about **1.33%**, equity volatility was below average, and macro stress indicators such as VIX and credit spreads were lower than usual. This supports a high-equity allocation with a meaningful gold sleeve.
- **Regime 1:** SPY still had a strong average monthly return of about **1.20%**, equity momentum was the strongest of the three regimes, and the macro profile looked more policy-supported. This supports the highest SPY weight, with some TLT retained as a diversifier.
- **Regime 2:** SPY had an average monthly return of about **-0.36%**, annualised equity volatility was much higher at about **24.65%**, drawdowns were deeper, and VIX/credit stress were elevated. This supports cutting equity exposure and spreading defensive exposure across TLT, GLD, and SHY.

This is still a human-designed allocation policy. The important point is that the weights are chosen from evidence observed in the regime interpretation, written down before the backtest, and kept simple enough to audit.

### Strategy constraints

The strategy is intentionally constrained before any performance backtest is run.

- **Long-only:** no short positions.
- **Fully invested:** weights across SPY, TLT, GLD, and SHY sum to 100%.
- **No leverage:** total portfolio exposure is capped at 100%.
- **Monthly decisions:** the strategy updates at the same frequency as the regime features.
- **Predefined regime weights:** the weights are chosen from the regime interpretation, not from test-period performance.
- **Execution lag in the later backtest:** the allocation signal formed at month-end should only be applied from the next month onward to avoid look-ahead bias.

These constraints make the rule easier to defend. If the strategy performs well later, the result is less likely to come from hidden leverage, excessive trading, or fitting the test period.

### Define regime labels and base allocations

The regime labels below come from the evidence in Part 6. They are not imposed before training the model.

The weights are deliberately simple. They are not optimised to maximise historical return. Instead, they express a clear investment view for each inferred environment:

- **Regime 0: Resilient risk-on.** Keep high equity exposure because SPY performed well, but retain a meaningful GLD sleeve because gold also had strong average returns in this state.
- **Regime 1: Policy-supported recovery.** Lean most heavily into equities because this state showed strong equity trend evidence, positive SPY returns, and a more supportive policy backdrop.
- **Regime 2: Stress.** Reduce SPY exposure and spread the defensive sleeve across TLT, GLD, and SHY. Long-duration Treasuries can help during growth shocks, but they may be less reliable when stress is driven by inflation or rising rates, so the defensive allocation is diversified.

In [ ]:
REGIME_LABELS = {
    0: "Resilient risk-on",
    1: "Policy-supported recovery",
    2: "Stress",
}

ASSET_COLUMNS = ["SPY", "TLT", "GLD", "SHY"]

# These are policy weights, not fitted model parameters.
# They are chosen before the backtest and should not be tuned on test-period returns.
regime_allocation_policy = pd.DataFrame(
    [
        {
            "regime_state": 0,
            "regime_label": REGIME_LABELS[0],
            "SPY": 0.70,
            "TLT": 0.10,
            "GLD": 0.15,
            "SHY": 0.05,
            "allocation_rationale": "High equity exposure with a gold sleeve, reflecting strong SPY and GLD returns with lower stress.",
        },
        {
            "regime_state": 1,
            "regime_label": REGIME_LABELS[1],
            "SPY": 0.75,
            "TLT": 0.15,
            "GLD": 0.05,
            "SHY": 0.05,
            "allocation_rationale": "Highest equity allocation, reflecting strong momentum and a policy-supported macro backdrop.",
        },
        {
            "regime_state": 2,
            "regime_label": REGIME_LABELS[2],
            "SPY": 0.25,
            "TLT": 0.25,
            "GLD": 0.25,
            "SHY": 0.25,
            "allocation_rationale": "Lower equity exposure and diversified defensive allocation during high-volatility stress periods.",
        },
    ]
)

# Check that each regime portfolio is fully invested.
regime_allocation_policy["weight_sum"] = regime_allocation_policy[ASSET_COLUMNS].sum(axis=1)

if not np.allclose(regime_allocation_policy["weight_sum"], 1.0):
    raise ValueError("Each regime allocation must sum to 100%.")

REGIME_ALLOCATION_POLICY_CACHE = PROCESSED_DATA_DIR / "regime_allocation_policy.csv"
regime_allocation_policy.to_csv(REGIME_ALLOCATION_POLICY_CACHE, index=False)

allocation_display = regime_allocation_policy.copy()
allocation_display[ASSET_COLUMNS + ["weight_sum"]] = allocation_display[ASSET_COLUMNS + ["weight_sum"]] * 100

display(allocation_display.round(1))

The allocation table is the main design object in this section. The most important thing to notice is that the stress regime does not simply move everything into TLT or cash. It reduces SPY exposure, but splits the defensive allocation across TLT, GLD, and SHY because different stress periods have different causes.

This makes the rule more cautious than assuming one defensive asset always works. It also keeps the strategy easy to explain: increase risk exposure when the model identifies stronger environments, and reduce equity concentration when the model identifies stress.

### Convert regime probabilities into portfolio weights

A useful feature of HMMs is that they produce probabilities, not only hard labels. That means the portfolio does not need to jump fully from one regime allocation to another.

Instead, the strategy uses probability-weighted allocation:

```text
Final allocation
= P(Regime 0) * Regime 0 weights
+ P(Regime 1) * Regime 1 weights
+ P(Regime 2) * Regime 2 weights
```

For example, if the model is 60% confident in a risk-on regime and 40% confident in a stress regime, the portfolio becomes a blend of those two policies. This is more consistent with the probabilistic nature of the HMM than hard-switching entirely based on the most likely state.

In [ ]:
probability_columns = {
    0: "state_0_probability",
    1: "state_1_probability",
    2: "state_2_probability",
}

missing_probability_columns = [
    column for column in probability_columns.values()
    if column not in regime_probabilities.columns
]

if missing_probability_columns:
    raise ValueError(f"Missing regime probability columns: {missing_probability_columns}")

# Arrange probability columns and regime weights in the same regime-state order.
regime_probability_matrix = regime_probabilities[
    [probability_columns[state] for state in sorted(probability_columns)]
].copy()
regime_probability_matrix.columns = sorted(probability_columns)

regime_weight_matrix = (
    regime_allocation_policy
    .set_index("regime_state")
    .loc[sorted(probability_columns), ASSET_COLUMNS]
)

# Matrix multiplication applies the probability-weighted allocation formula month by month.
strategy_target_weights = regime_probability_matrix @ regime_weight_matrix
strategy_target_weights.columns = ASSET_COLUMNS

# Small floating-point errors can appear after matrix multiplication, so normalise the rows.
strategy_target_weights = strategy_target_weights.div(
    strategy_target_weights.sum(axis=1),
    axis=0,
)

strategy_target_weights["most_likely_state"] = regime_probabilities["most_likely_state"].astype(int)
strategy_target_weights["regime_label"] = strategy_target_weights["most_likely_state"].map(REGIME_LABELS)
strategy_target_weights["weight_sum"] = strategy_target_weights[ASSET_COLUMNS].sum(axis=1)

if not np.allclose(strategy_target_weights["weight_sum"], 1.0):
    raise ValueError("Monthly strategy target weights must sum to 100%.")

STRATEGY_TARGET_WEIGHTS_CACHE = PROCESSED_DATA_DIR / "strategy_target_weights.csv"
strategy_target_weights.to_csv(STRATEGY_TARGET_WEIGHTS_CACHE, index_label="date")

print(f"Strategy target weights saved to: {STRATEGY_TARGET_WEIGHTS_CACHE.relative_to(PROJECT_ROOT)}")
print(f"Sample period: {strategy_target_weights.index.min().date()} to {strategy_target_weights.index.max().date()}")

display(strategy_target_weights.head().round(4))

This code converts the HMM probabilities into target portfolio weights. Each row is one month-end signal date. The weights shown here are target allocations, not realised strategy returns. In months where the model assigns almost all probability to one regime, the target weights will look exactly like that regime's base allocation. In more uncertain months, the weights become a blend across regimes.

The later backtest should apply these signals with a one-month lag. In plain language, if the model observes data up to the end of March, the portfolio should only trade for April. That timing rule matters because it prevents the strategy from accidentally using information before an investor would have known it.

### Inspect representative allocation signals

Before calculating performance, it is helpful to check whether the allocation rule behaves sensibly in familiar historical periods. This is a diagnostic step to confirm that the rule responds in the intended direction.

In [ ]:
representative_dates = pd.to_datetime([
    "2008-10-31",  # Global Financial Crisis stress period
    "2013-12-31",  # Policy-supported recovery period
    "2016-01-31",  # Market stress around global growth and oil concerns
    "2019-01-31",  # Uncertain transition after late-2018 volatility
    "2020-03-31",  # COVID shock
])

available_representative_dates = [
    date for date in representative_dates
    if date in strategy_target_weights.index
]

representative_allocations = strategy_target_weights.loc[
    available_representative_dates,
    ASSET_COLUMNS + ["most_likely_state", "regime_label"],
].copy()

representative_allocations[ASSET_COLUMNS] = representative_allocations[ASSET_COLUMNS] * 100

display(representative_allocations.round(1))

The representative dates show the basic behaviour of the rule. Stress-heavy periods should generally show lower SPY exposure and higher defensive exposure. Recovery or resilient-risk periods should generally show higher SPY exposure.

The January 2019 example is especially useful because the model was close to split between Regime 0 and Regime 2, so the final allocation is blended rather than a clean round-number regime portfolio.

### Inspect uncertain regime probabilities

The previous table includes hand-picked historical dates. To see the probability-weighted rule more directly, we can also search for months where the HMM was least certain.

To keep this diagnostic separate from later test-period evaluation, the search below only uses dates up to the end of the validation period.

In [ ]:
probability_display_columns = [
    probability_columns[state]
    for state in sorted(probability_columns)
]

# Find months where the model's highest regime probability was lowest.
# These are the months where probability-weighted allocation matters most.
uncertain_probability_window = regime_probabilities.loc[
    regime_probabilities.index <= VALIDATION_END,
    probability_display_columns + ["most_likely_state"],
].copy()

uncertain_probability_window["max_regime_probability"] = uncertain_probability_window[
    probability_display_columns
].max(axis=1)

uncertain_dates = (
    uncertain_probability_window
    .sort_values("max_regime_probability")
    .head(5)
    .index
)

uncertain_allocation_examples = pd.DataFrame(index=uncertain_dates)

for state in sorted(probability_columns):
    uncertain_allocation_examples[f"P(Regime {state})"] = (
        regime_probabilities.loc[uncertain_dates, probability_columns[state]] * 100
    )

for asset in ASSET_COLUMNS:
    uncertain_allocation_examples[f"{asset} weight"] = (
        strategy_target_weights.loc[uncertain_dates, asset] * 100
    )

uncertain_allocation_examples["most_likely_state"] = strategy_target_weights.loc[
    uncertain_dates,
    "most_likely_state",
]
uncertain_allocation_examples["regime_label"] = strategy_target_weights.loc[
    uncertain_dates,
    "regime_label",
]

# Show percentages so the blended allocations are easy to read.
display(uncertain_allocation_examples.round(1))

This table shows why probability-weighting is useful. When the model is uncertain, the final portfolio is not forced into a single regime's allocation. For example, a month split between resilient risk-on and stress will produce a portfolio with less SPY than a pure risk-on month, but more SPY than a pure stress month.

The purpose of this check is directional: before backtesting, the allocation should respond intuitively to both confident and uncertain regime estimates.

### What has been decided in Part 7

Part 7 defines the bridge between regime detection and portfolio construction:

- The selected HMM remains the 3-state model from Part 5.
- The regime labels come from the evidence in Part 6.
- Each regime is mapped to a clear long-only allocation across SPY, TLT, GLD, and SHY.
- Monthly target weights are calculated using regime probabilities rather than hard switching.
- The allocation rule is saved locally for the next section.

The next section can now evaluate the rule properly. That means applying a realistic signal lag, adding transaction costs, comparing against benchmarks, and checking whether the regime-aware allocation improves risk-adjusted performance without relying on hindsight.

## 8. Walk-Forward Backtest

The previous sections trained and interpreted the HMM. This section tests whether the regime idea can be turned into a realistic historical portfolio simulation.

The key principle is simple: at each month-end, the strategy can only use information that would have been available at that time. It then chooses the allocation for the next month. The model still works at a monthly frequency, but portfolio performance is measured using daily ETF returns.

In plain language:

```text
End of January:
    train the scaler and HMM using data available up to January
    estimate January's regime probabilities
    decide February's ETF weights

Every trading day in February:
    hold those weights and record daily portfolio returns

End of February:
    repeat the process for March
```

This is more realistic than fitting the HMM once on the full sample and then pretending those fitted regimes were known through history.

### Backtest design choices

This walk-forward test uses the following design choices:

- **Expanding training window:** every monthly refit uses all available feature history up to that signal date.
- **First signal date:** the first signal is formed at the end of the original training period, so the daily simulation begins in January 2019.
- **Validation-era vs test-period evidence:** 2019-2021 is still validation-era walk-forward evidence because that period was used earlier for model selection. The cleaner held-out evaluation starts in 2022.
- **Monthly signal, daily accounting:** the HMM produces monthly target weights, but the equity curve is built from daily ETF returns.
- **State relabelling:** HMM state numbers can switch between refits, so each refit maps raw states back to the human-readable regimes using only past data.
- **Transaction costs:** monthly rebalances include a simple proportional all-in trading friction based on portfolio turnover.
- **Benchmarks:** the main comparison uses SPY buy-and-hold and a static 60/40 SPY/TLT portfolio.

One important point: this section does **not** reuse the full-sample target weights created in Part 7. Those weights were useful for explaining the allocation policy. The walk-forward backtest recreates the signal month by month.

In [ ]:
from io import BytesIO

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image


def display_matplotlib_figure(fig):
    """Display a Matplotlib figure as a PNG so exported HTML includes the chart."""
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
from sklearn.preprocessing import StandardScaler
from hmmlearn.hmm import GaussianHMM

DAILY_PRICE_CACHE = RAW_DATA_DIR / "tiingo_etf_prices.csv"

if not DAILY_PRICE_CACHE.exists():
    raise FileNotFoundError(
        "Daily Tiingo price cache was not found. Run the feature engineering section first."
    )

daily_prices = pd.read_csv(
    DAILY_PRICE_CACHE,
    parse_dates=["date"],
    index_col="date",
).sort_index()

daily_prices = daily_prices[ASSET_COLUMNS].dropna()
daily_asset_returns = daily_prices.pct_change().dropna()

WALK_FORWARD_N_STATES = 3
WALK_FORWARD_FIRST_SIGNAL_DATE = TRAIN_END
WALK_FORWARD_RANDOM_SEEDS = range(10)
TRANSACTION_COST_BPS = 5
TRANSACTION_COST_RATE = TRANSACTION_COST_BPS / 10_000

walk_forward_signal_dates = feature_matrix.index[
    feature_matrix.index >= WALK_FORWARD_FIRST_SIGNAL_DATE
]

print(f"Daily price sample: {daily_prices.index.min().date()} to {daily_prices.index.max().date()}")
print(f"First walk-forward signal date: {walk_forward_signal_dates.min().date()}")
print(f"Number of monthly signal dates: {len(walk_forward_signal_dates)}")
print(f"Transaction cost assumption: {TRANSACTION_COST_BPS} bps per unit of turnover")

display(daily_prices.tail())

The setup cell loads the daily ETF price cache created earlier from Tiingo. The monthly model uses the engineered `feature_matrix`, while the daily return series is used only for portfolio accounting.

The first signal is formed at **2018-12-31**, which means the first realised portfolio returns start in **January 2019**. The full simulation therefore covers both the validation-era period and the later test period. When interpreting results, the 2022 onward table should receive more weight because it is the cleaner held-out evaluation.

### Helper functions for walk-forward model fitting

The next functions do three jobs.

First, they fit a fresh scaler and HMM using only past data. Second, they assign raw HMM states back to regime labels such as `Stress`, because the raw state numbers are temporary model labels rather than permanent regime names. Third, they convert the current regime probabilities into ETF weights using the allocation policy from Part 7.

In [ ]:
def fit_hmm_walk_forward_candidate(n_states, random_seed, X):
    """Fit one HMM candidate for a walk-forward refit."""
    rng = np.random.default_rng(random_seed)

    model = GaussianHMM(
        n_components=n_states,
        covariance_type="diag",
        n_iter=N_ITER,
        tol=TOLERANCE,
        min_covar=1e-3,
        random_state=random_seed,
        init_params="",
    )

    model.startprob_ = np.repeat(1 / n_states, n_states)

    off_diagonal_probability = 0.05 / (n_states - 1)
    model.transmat_ = np.full((n_states, n_states), off_diagonal_probability)
    np.fill_diagonal(model.transmat_, 0.95)

    initial_mean_rows = rng.choice(len(X), size=n_states, replace=False)
    model.means_ = X[initial_mean_rows].copy()
    model.covars_ = np.tile(np.var(X, axis=0) + 1e-3, (n_states, 1))

    model.fit(X)
    return model


def fit_walk_forward_hmm(training_features):
    """Fit a scaler and keep the best HMM candidate across several random starts."""
    scaler = StandardScaler()
    X_train = scaler.fit_transform(training_features)

    best_model = None
    best_log_likelihood = -np.inf

    for seed in WALK_FORWARD_RANDOM_SEEDS:
        model = fit_hmm_walk_forward_candidate(
            n_states=WALK_FORWARD_N_STATES,
            random_seed=seed,
            X=X_train,
        )
        train_log_likelihood = model.score(X_train)

        if train_log_likelihood > best_log_likelihood:
            best_log_likelihood = train_log_likelihood
            best_model = model

    return scaler, best_model, X_train, best_log_likelihood


def zscore_across_states(summary_table):
    """Convert state-level metrics into z-scores across states for scoring."""
    state_std = summary_table.std(ddof=0).replace(0, np.nan)
    zscores = (summary_table - summary_table.mean()) / state_std
    return zscores.fillna(0)


def map_raw_states_to_regime_labels(training_features, training_returns, raw_states):
    """Map raw HMM state numbers to evidence-based regime labels using past data only."""
    state_data = training_features.join(
        training_returns[["SPY"]].rename(columns={"SPY": "asset_spy_return"}),
        how="inner",
    ).copy()
    state_data["raw_state"] = raw_states

    state_summary = (
        state_data
        .groupby("raw_state")
        .agg(
            asset_spy_return=("asset_spy_return", "mean"),
            spy_momentum_6m=("spy_momentum_6m", "mean"),
            spy_momentum_12m=("spy_momentum_12m", "mean"),
            spy_drawdown=("spy_drawdown", "mean"),
            spy_realized_vol_3m=("spy_realized_vol_3m", "mean"),
            vix_level=("vix_level", "mean"),
            credit_spread_baa10y=("credit_spread_baa10y", "mean"),
            term_spread_10y2y=("term_spread_10y2y", "mean"),
            term_spread_10y3m=("term_spread_10y3m", "mean"),
            fed_funds_rate=("fed_funds_rate", "mean"),
        )
        .reindex(range(WALK_FORWARD_N_STATES))
    )

    # If a state is extremely rare in a refit, fill missing diagnostics with cross-state averages.
    state_summary = state_summary.fillna(state_summary.mean())
    state_zscores = zscore_across_states(state_summary)

    # Stress is defined by weak equity returns, deeper drawdowns, higher volatility, and higher risk stress.
    stress_score = (
        -state_zscores["asset_spy_return"]
        -state_zscores["spy_drawdown"]
        + state_zscores["spy_realized_vol_3m"]
        + state_zscores["vix_level"]
        + state_zscores["credit_spread_baa10y"]
    )
    stress_state = int(stress_score.idxmax())

    remaining_states = [state for state in range(WALK_FORWARD_N_STATES) if state != stress_state]

    # Recovery is defined by stronger equity momentum and a more policy-supported macro backdrop.
    recovery_score = (
        state_zscores["spy_momentum_6m"]
        + state_zscores["spy_momentum_12m"]
        + state_zscores["term_spread_10y2y"]
        + state_zscores["term_spread_10y3m"]
        - state_zscores["fed_funds_rate"]
    )
    recovery_state = int(recovery_score.loc[remaining_states].idxmax())

    risk_on_state = [
        state for state in range(WALK_FORWARD_N_STATES)
        if state not in [stress_state, recovery_state]
    ][0]

    raw_state_to_label = {
        risk_on_state: "Resilient risk-on",
        recovery_state: "Policy-supported recovery",
        stress_state: "Stress",
    }

    mapping_diagnostics = pd.DataFrame(
        {
            "raw_state": list(raw_state_to_label.keys()),
            "regime_label": list(raw_state_to_label.values()),
            "stress_score": [stress_score.loc[state] for state in raw_state_to_label],
            "recovery_score": [recovery_score.loc[state] for state in raw_state_to_label],
        }
    )

    return raw_state_to_label, mapping_diagnostics


def convert_probabilities_to_policy_weights(raw_probabilities, raw_state_to_label, policy_table):
    """Convert raw HMM state probabilities into ETF weights using canonical regime labels."""
    label_probabilities = pd.Series(0.0, index=policy_table["regime_label"])

    for raw_state, probability in enumerate(raw_probabilities):
        regime_label = raw_state_to_label[raw_state]
        label_probabilities.loc[regime_label] += probability

    policy_by_label = policy_table.set_index("regime_label")[ASSET_COLUMNS]
    target_weights = label_probabilities @ policy_by_label

    return label_probabilities, target_weights

The state relabelling step is crucial. HMM state numbers are not stable names. A state called `0` in one monthly refit might look like stress, while state `0` in another refit might look like risk-on. The number is closer to a temporary seat number than a permanent regime name.

In a live research process, a human analyst could inspect the states after every refit. In a backtest, that manual interpretation is not practical, so the code uses a consistent heuristic instead. It scores each raw state using only data available at that rebalance date. The stress state is the one with weaker SPY returns, worse drawdowns, higher volatility, higher VIX, and wider credit spreads. Among the remaining states, the recovery state is the one with stronger equity momentum and a more policy-supported macro backdrop. The remaining state is labelled resilient risk-on.

### Generate monthly walk-forward signals

The loop below is the core of the walk-forward backtest. For each month-end signal date, it trains a new scaler and HMM on the expanding history, infers the latest regime probabilities, maps those probabilities to regime labels, and records the ETF allocation for the following month.

In [ ]:
walk_forward_signal_rows = []
walk_forward_mapping_rows = []

for signal_date in walk_forward_signal_dates:
    training_features = feature_matrix.loc[:signal_date].copy()
    training_returns = asset_returns.loc[training_features.index].copy()

    scaler, model, X_train, train_log_likelihood = fit_walk_forward_hmm(training_features)
    raw_training_states = model.predict(X_train)

    raw_state_to_label, mapping_diagnostics = map_raw_states_to_regime_labels(
        training_features=training_features,
        training_returns=training_returns,
        raw_states=raw_training_states,
    )

    # Use the posterior probability of the final training month.
    # Passing only the current row would ignore the Markov time sequence.
    raw_probabilities = model.predict_proba(X_train)[-1]

    label_probabilities, target_weights = convert_probabilities_to_policy_weights(
        raw_probabilities=raw_probabilities,
        raw_state_to_label=raw_state_to_label,
        policy_table=regime_allocation_policy,
    )

    most_likely_label = label_probabilities.idxmax()

    signal_row = {
        "signal_date": signal_date,
        "execution_month": (signal_date.to_period("M") + 1).strftime("%Y-%m"),
        "training_months": len(training_features),
        "train_log_likelihood_per_month": train_log_likelihood / len(training_features),
        "most_likely_regime": most_likely_label,
    }

    for label, probability in label_probabilities.items():
        signal_row[f"{label}_probability"] = probability

    for asset, weight in target_weights.items():
        signal_row[asset] = weight

    walk_forward_signal_rows.append(signal_row)

    mapping_diagnostics["signal_date"] = signal_date
    walk_forward_mapping_rows.append(mapping_diagnostics)

walk_forward_signals = (
    pd.DataFrame(walk_forward_signal_rows)
    .set_index("signal_date")
    .sort_index()
)

walk_forward_state_mapping = pd.concat(walk_forward_mapping_rows, ignore_index=True)

WALK_FORWARD_SIGNAL_CACHE = PROCESSED_DATA_DIR / "walk_forward_monthly_signals.csv"
WALK_FORWARD_STATE_MAPPING_CACHE = PROCESSED_DATA_DIR / "walk_forward_state_mapping.csv"

walk_forward_signals.to_csv(WALK_FORWARD_SIGNAL_CACHE, index_label="signal_date")
walk_forward_state_mapping.to_csv(WALK_FORWARD_STATE_MAPPING_CACHE, index=False)

print(f"Saved walk-forward monthly signals to: {WALK_FORWARD_SIGNAL_CACHE.relative_to(PROJECT_ROOT)}")
print(f"Signal period: {walk_forward_signals.index.min().date()} to {walk_forward_signals.index.max().date()}")

display(walk_forward_signals.head().round(4))
display(walk_forward_signals.tail().round(4))

The output above is the monthly signal table. It is not the daily performance yet. Each row is a month-end decision point: what regime the model infers, how confident it is, and what allocation will be used in the following month.

For example, a signal formed at the end of December 2018 is used for January 2019 returns. This one-month timing discipline is what prevents look-ahead bias.

### Inspect walk-forward signal behaviour

Before calculating returns, we should check that the walk-forward signal stream looks reasonable: which regimes were most common, what the average allocation looked like, and how much uncertainty the model had through time.

In [ ]:
walk_forward_regime_counts = walk_forward_signals["most_likely_regime"].value_counts().rename("months")
walk_forward_regime_summary = pd.DataFrame(
    {
        "months": walk_forward_regime_counts,
        "share_of_signals": walk_forward_regime_counts / len(walk_forward_signals),
    }
)

walk_forward_average_weights = walk_forward_signals[ASSET_COLUMNS].mean().to_frame("average_weight")

probability_columns_walk_forward = [
    column for column in walk_forward_signals.columns
    if column.endswith("_probability")
]
walk_forward_confidence = walk_forward_signals[probability_columns_walk_forward].max(axis=1)
confidence_summary = walk_forward_confidence.describe().to_frame("highest_regime_probability")

print("Most likely regime frequency")
display(walk_forward_regime_summary.round(4))

print("Average walk-forward target weights")
display((walk_forward_average_weights * 100).round(2))

print("Signal confidence summary")
display(confidence_summary.round(4))

The walk-forward signal is not constant. Across 90 monthly signal dates, the strategy classifies 58 months as resilient risk-on, 16 months as policy-supported recovery, and 16 months as stress. This gives an average target allocation of about 62.9% SPY, 13.6% TLT, 15.0% GLD, and 8.6% SHY.

The confidence summary is also informative. In this walk-forward run, the highest regime probability is usually extremely close to 1. That means the probability-weighted rule is still implemented correctly, but in practice it often behaves like a hard-switching rule because the fitted HMM is very confident about the most likely state.

This does not make probability-weighting unnecessary. It remains the more general design for an HMM because the model naturally outputs probabilities. If a future model specification produces softer regime probabilities, the same allocation rule can blend exposures without changing the strategy logic.

### Convert monthly target weights into daily portfolio returns

The HMM makes monthly allocation decisions, but investors experience returns day by day. The next code block applies each target allocation to the daily ETF returns in the following month.

Within each month, the portfolio starts at the target weights and then drifts naturally with asset returns. At the next month, the portfolio is rebalanced to the new target weights, and transaction costs are charged based on turnover.

The baseline transaction cost is **5 bps**, which means **0.05%** per unit of turnover. This is not meant to represent only broker commission. It is a small all-in trading friction allowance for bid-ask spread, slippage, foreign exchange or platform frictions, and the general fact that real execution is not perfectly frictionless.

In [ ]:
def build_target_weights_by_execution_month(monthly_signals, asset_columns):
    """Index monthly target weights by the month in which they are applied."""
    target_weights = monthly_signals[asset_columns].copy()
    target_weights.index = monthly_signals["execution_month"].apply(lambda value: pd.Period(value, freq="M"))
    target_weights.index.name = "execution_month"
    return target_weights


def calculate_monthly_rebalanced_daily_returns(daily_returns, target_weights_by_month, cost_rate=0.0):
    """Calculate daily portfolio returns from monthly target weights."""
    portfolio_return_pieces = []
    turnover_rows = []
    previous_end_weights = None

    for execution_month, target_weights in target_weights_by_month.iterrows():
        month_returns = daily_returns.loc[
            daily_returns.index.to_period("M") == execution_month,
            target_weights.index,
        ].dropna()

        if month_returns.empty:
            continue

        target_weights = target_weights.astype(float)
        target_weights = target_weights / target_weights.sum()

        if previous_end_weights is None:
            turnover = 0.0
        else:
            turnover = (target_weights - previous_end_weights).abs().sum()

        # Let holdings drift within the month instead of forcing daily rebalancing.
        asset_growth = (1 + month_returns).cumprod()
        gross_portfolio_value = asset_growth.mul(target_weights, axis=1).sum(axis=1)
        gross_portfolio_returns = gross_portfolio_value.pct_change()
        gross_portfolio_returns.iloc[0] = month_returns.iloc[0].mul(target_weights).sum()

        transaction_cost = turnover * cost_rate
        net_portfolio_returns = gross_portfolio_returns.copy()
        net_portfolio_returns.iloc[0] = net_portfolio_returns.iloc[0] - transaction_cost

        portfolio_return_pieces.append(net_portfolio_returns)

        end_asset_values = target_weights * asset_growth.iloc[-1]
        previous_end_weights = end_asset_values / end_asset_values.sum()

        turnover_rows.append(
            {
                "execution_month": execution_month,
                "turnover": turnover,
                "transaction_cost": transaction_cost,
            }
        )

    portfolio_returns = pd.concat(portfolio_return_pieces).sort_index()
    turnover_table = pd.DataFrame(turnover_rows).set_index("execution_month")
    return portfolio_returns, turnover_table


def make_static_monthly_weights(target_weights_by_month, weights):
    """Create a benchmark weight table with the same execution months as the strategy."""
    static_weights = pd.DataFrame(index=target_weights_by_month.index, columns=ASSET_COLUMNS, dtype=float)
    for asset in ASSET_COLUMNS:
        static_weights[asset] = weights.get(asset, 0.0)
    return static_weights


strategy_weights_by_month = build_target_weights_by_execution_month(
    monthly_signals=walk_forward_signals,
    asset_columns=ASSET_COLUMNS,
)

strategy_daily_returns, strategy_turnover = calculate_monthly_rebalanced_daily_returns(
    daily_returns=daily_asset_returns,
    target_weights_by_month=strategy_weights_by_month,
    cost_rate=TRANSACTION_COST_RATE,
)

benchmark_weight_tables = {
    "SPY buy-and-hold": make_static_monthly_weights(strategy_weights_by_month, {"SPY": 1.0}),
    "60/40 SPY/TLT": make_static_monthly_weights(strategy_weights_by_month, {"SPY": 0.60, "TLT": 0.40}),
}

benchmark_daily_returns = {}
benchmark_turnover = {}

for benchmark_name, benchmark_weights in benchmark_weight_tables.items():
    returns, turnover = calculate_monthly_rebalanced_daily_returns(
        daily_returns=daily_asset_returns,
        target_weights_by_month=benchmark_weights,
        cost_rate=TRANSACTION_COST_RATE,
    )
    benchmark_daily_returns[benchmark_name] = returns
    benchmark_turnover[benchmark_name] = turnover

walk_forward_daily_returns = pd.DataFrame({
    "HMM regime strategy": strategy_daily_returns,
    **benchmark_daily_returns,
}).dropna()

test_period_daily_returns = walk_forward_daily_returns.loc[
    walk_forward_daily_returns.index > VALIDATION_END
].copy()

WALK_FORWARD_DAILY_RETURNS_CACHE = PROCESSED_DATA_DIR / "walk_forward_daily_returns.csv"
WALK_FORWARD_TURNOVER_CACHE = PROCESSED_DATA_DIR / "walk_forward_strategy_turnover.csv"

walk_forward_daily_returns.to_csv(WALK_FORWARD_DAILY_RETURNS_CACHE, index_label="date")
strategy_turnover.to_csv(WALK_FORWARD_TURNOVER_CACHE, index_label="execution_month")

print(f"Daily backtest period: {walk_forward_daily_returns.index.min().date()} to {walk_forward_daily_returns.index.max().date()}")
print(f"Number of daily observations: {len(walk_forward_daily_returns)}")
print(f"Average monthly strategy turnover: {strategy_turnover['turnover'].mean():.2%}")

display(walk_forward_daily_returns.head().round(4))
display(strategy_turnover.head().round(4))

This step creates the daily return series that will be used for performance analysis. The strategy and benchmarks are placed on the same daily date index, which makes the later equity curve and metrics directly comparable.

The main benchmarks are SPY buy-and-hold and 60/40 SPY/TLT. SPY tests whether the strategy adds value compared with simply holding equities. 60/40 tests whether it adds value compared with a classic static diversified allocation. An equal-weight SPY/TLT/GLD/SHY portfolio can still be useful as a robustness check later, but it is excluded from the main chart because it is less realistic and makes the core comparison less readable.

The turnover table records how much the strategy had to trade at each monthly rebalance. Turnover matters because a strategy that improves returns only by trading aggressively may look less attractive after transaction costs.

In [ ]:
def calculate_max_drawdown(return_series):
    """Calculate the maximum peak-to-trough drawdown for a return series."""
    equity_curve = (1 + return_series).cumprod()
    running_peak = equity_curve.cummax()
    drawdown = equity_curve / running_peak - 1
    return drawdown.min()


def calculate_performance_metrics(daily_return_table, annualization_factor=252):
    """Calculate common daily-return performance metrics for each strategy."""
    metric_rows = []

    for strategy_name, returns in daily_return_table.items():
        returns = returns.dropna()
        total_return = (1 + returns).prod() - 1
        annualized_return = (1 + total_return) ** (annualization_factor / len(returns)) - 1
        annualized_volatility = returns.std() * np.sqrt(annualization_factor)
        sharpe_ratio = np.nan
        if annualized_volatility != 0:
            sharpe_ratio = returns.mean() / returns.std() * np.sqrt(annualization_factor)

        max_drawdown = calculate_max_drawdown(returns)
        calmar_ratio = np.nan
        if max_drawdown != 0:
            calmar_ratio = annualized_return / abs(max_drawdown)

        metric_rows.append(
            {
                "strategy": strategy_name,
                "total_return": total_return,
                "annualized_return": annualized_return,
                "annualized_volatility": annualized_volatility,
                "sharpe_ratio": sharpe_ratio,
                "max_drawdown": max_drawdown,
                "calmar_ratio": calmar_ratio,
            }
        )

    return pd.DataFrame(metric_rows).set_index("strategy")



def format_metric_annotation(metrics_table):
    """Create a compact chart annotation from a performance metrics table."""
    lines = ["Ann Ret | Ann Vol | Sharpe | Max DD"]

    for strategy_name, row in metrics_table.iterrows():
        lines.append(
            f"{strategy_name}: "
            f"{row['annualized_return']:.1%} | "
            f"{row['annualized_volatility']:.1%} | "
            f"{row['sharpe_ratio']:.2f} | "
            f"{row['max_drawdown']:.1%}"
        )

    return "\n".join(lines)


def add_metric_annotation(ax, metrics_table):
    """Place compact performance metrics inside an equity chart."""
    ax.text(
        0.02,
        0.98,
        format_metric_annotation(metrics_table),
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8.5,
        bbox={
            "boxstyle": "round,pad=0.45",
            "facecolor": "white",
            "edgecolor": "#d0d0d0",
            "alpha": 0.88,
        },
    )

### Plot equity curves

The equity curve shows the growth of one dollar invested at the start of the full walk-forward simulation. This makes it easy to compare the path of the regime strategy against the two main static alternatives.

In [ ]:
walk_forward_equity_curves = (1 + walk_forward_daily_returns).cumprod()
walk_forward_metrics_for_chart = calculate_performance_metrics(walk_forward_daily_returns)

fig, ax = plt.subplots(figsize=(12, 6))
for column in walk_forward_equity_curves.columns:
    ax.plot(walk_forward_equity_curves.index, walk_forward_equity_curves[column], label=column)

ax.set_title("Walk-Forward Backtest Equity Curves")
ax.set_ylabel("Growth of $1")
ax.set_xlabel("Date")
ax.legend(loc="lower right")
add_metric_annotation(ax, walk_forward_metrics_for_chart)
ax.grid(True, alpha=0.3)

display_matplotlib_figure(fig)
plt.close(fig)

The equity curve is useful, but it should not be read alone. A strategy can end with a high final value while still having an uncomfortable drawdown path. That is why the next step calculates risk-adjusted metrics and drawdowns.

### Plot held-out test-period equity curves

The full equity curve is useful for seeing the whole walk-forward simulation, but the cleaner evaluation starts after the validation window. The next chart resets each strategy to `$1` at the start of 2022 so the held-out test-period paths can be compared directly.

In [ ]:
test_period_equity_curves = (1 + test_period_daily_returns).cumprod()
test_period_metrics_for_chart = calculate_performance_metrics(test_period_daily_returns)

fig, ax = plt.subplots(figsize=(12, 6))
for column in test_period_equity_curves.columns:
    ax.plot(test_period_equity_curves.index, test_period_equity_curves[column], label=column)

ax.set_title("Held-Out Test Period Equity Curves: 2022 Onward")
ax.set_ylabel("Growth of $1")
ax.set_xlabel("Date")
ax.legend(loc="lower right")
add_metric_annotation(ax, test_period_metrics_for_chart)
ax.grid(True, alpha=0.3)

display_matplotlib_figure(fig)
plt.close(fig)

This chart is stricter than the full simulation chart because it begins after the validation period. It answers a narrower question: after the model structure and allocation rule were already chosen, how did the strategy behave in the later held-out period?

### Calculate performance metrics

The next table summarizes return and risk. Since this is a daily return backtest, annualized return and volatility use a 252-trading-day convention.

In [ ]:
walk_forward_metrics = calculate_performance_metrics(walk_forward_daily_returns)
test_period_metrics = calculate_performance_metrics(test_period_daily_returns)

WALK_FORWARD_METRICS_CACHE = PROCESSED_DATA_DIR / "walk_forward_performance_metrics.csv"
TEST_PERIOD_METRICS_CACHE = PROCESSED_DATA_DIR / "walk_forward_test_period_metrics.csv"

walk_forward_metrics.to_csv(WALK_FORWARD_METRICS_CACHE)
test_period_metrics.to_csv(TEST_PERIOD_METRICS_CACHE)

print("Full walk-forward period metrics")
display(walk_forward_metrics.round(4))

print("Held-out test period metrics from 2022 onward")
display(test_period_metrics.round(4))

The full walk-forward simulation table covers the period from 2019 onward. This includes both validation-era evidence from 2019-2021 and held-out test-period evidence from 2022 onward. Over the full simulation, the HMM regime strategy earns an annualized return of about 12.7%, below SPY's 17.9%, but with lower annualized volatility of about 12.9% versus SPY's 19.5%. Its Sharpe ratio is about 1.00, slightly higher than SPY's 0.94, and its maximum drawdown is smaller at about -23.5% versus SPY's -33.7%.

The held-out test-period table is especially important because it lines up with the original train/validation/test split. From 2022 onward, the HMM strategy again trails SPY on annualized return, 11.7% versus 12.7%, but has lower volatility, 12.8% versus 17.6%, and a higher Sharpe ratio, 0.93 versus 0.77. This is consistent with the project's risk-aware objective: the strategy is not simply trying to beat SPY on raw return.

### Inspect drawdowns and turnover

Because this project is motivated by risk-aware allocation, drawdowns are just as important as returns. The next plot shows the drawdown path for each strategy.

In [ ]:
walk_forward_drawdowns = walk_forward_equity_curves / walk_forward_equity_curves.cummax() - 1

fig, ax = plt.subplots(figsize=(12, 5))
for column in walk_forward_drawdowns.columns:
    ax.plot(walk_forward_drawdowns.index, walk_forward_drawdowns[column], label=column)

ax.set_title("Walk-Forward Backtest Drawdowns")
ax.set_ylabel("Drawdown")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)

display_matplotlib_figure(fig)
plt.close(fig)

turnover_summary = strategy_turnover.describe().loc[["mean", "50%", "max"]]
print("Strategy turnover summary")
display(turnover_summary.round(4))

### Plot held-out test-period drawdowns

The held-out drawdown chart focuses only on the 2022 onward period. This makes it easier to inspect the period where the strategy should be judged most strictly.

In [ ]:
test_period_drawdowns = test_period_equity_curves / test_period_equity_curves.cummax() - 1

fig, ax = plt.subplots(figsize=(12, 5))
for column in test_period_drawdowns.columns:
    ax.plot(test_period_drawdowns.index, test_period_drawdowns[column], label=column)

ax.set_title("Held-Out Test Period Drawdowns: 2022 Onward")
ax.set_ylabel("Drawdown")
ax.set_xlabel("Date")
ax.legend()
ax.grid(True, alpha=0.3)

display_matplotlib_figure(fig)
plt.close(fig)

This view shows why the 2022 onward period deserves separate attention. The regime strategy reduces volatility relative to SPY, but the drawdown improvement is more modest than in the full 2019 onward simulation. That suggests the strategy helps, but does not fully solve inflation/rate-driven stress.

The drawdown chart shows whether the regime strategy actually reduced painful losses, rather than only changing the final return. In the full walk-forward simulation, the HMM strategy's maximum drawdown is about -23.5%, compared with -33.7% for SPY and -27.7% for the 60/40 benchmark.

Turnover is generally modest. The strategy's average monthly turnover is about 9.0%, the median is about 1.7%, and the maximum is about 100.8% during a major regime shift. This means transaction costs matter, but the strategy is not constantly replacing the whole portfolio every month.

### What this backtest does and does not prove

This section gives a more realistic historical simulation than the earlier full-sample regime diagnostics. The HMM is refit month by month, state labels are reassigned using past data only, and realised performance is measured from daily returns after the signal date.

However, this is still a research backtest, not proof that the strategy will work live. The allocation policy was designed after studying the project data, so the results should be interpreted as evidence from a disciplined historical simulation rather than a production trading claim.

The next step is to interpret the benchmark comparison carefully: where the regime strategy helped, where it failed, whether drawdowns improved, and whether the improvement is meaningful after costs.